# V23: LOO Quality Maximization + Replogle Augmentation

**Strategy:** Improve LOO synthetic perturbation quality through 4 smarter cell-selection methods, plus add real Replogle K562 knockouts as training data augmentation.

**Base:** V20 RUN=3 (Blind MLP + cosine_light + 3-tier PB + full LOO, **LB=4.08996**)
**Target:** LB > 4.15

### The 7 Runs

| Run | LOO Method | Replogle | Key Idea |
|-----|-----------|----------|----------|
| 0 | Baseline (5th percentile) | No | Reference — V20 RUN=3 config |
| 1 | NN-Matched | No | Match low-G cells to similar normal-G cells via PCA |
| 2 | Residual-Based | No | Select cells where G is lower than expected (regression residuals) |
| 3 | Adaptive Percentile | No | Per-gene GMM: bimodal → natural "off" pop, unimodal → 2nd pctile |
| 4 | Expression-Weighted | No | Continuous weighting, no hard cutoff |
| 5 | Replogle Augmentation | Yes | ~2000+ real K562 CRISPRi knockouts as training samples |
| 6 | Best LOO + Replogle | Yes | Combine winning LOO + Replogle |

### What is FROZEN (same as V20 RUN=3)
- **Architecture:** V17 LightMLP (Embedding → 2-layer MLP → 5127 output, H=384)
- **Loss:** cosine_light (λ=0.08, cos_right=0.0405)
- **3-Tier PseudoBulk:** T1=10 full, T2=per-channel, T3=5 half-cell
- **LOO for ALL 5127 genes, 3x resamples** (proven best LB)
- **No Hard Thresholding** (V21 proved harmful)
- **No Mixup, No Per-Channel LOO** (V22 proved harmful)
- **KO Correction:** knockout_multiplier=1.0

### Checkpoint/Resume
All runs checkpoint at fold level to `/content/drive/MyDrive/myllia_v23/`. Crashes resume from last completed fold.

In [ ]:
# ============================================================
# CELL 1: Setup & Drive Mount + V23 Checkpoint Directory
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/Myllia_Challenge'
DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
V23_DIR = '/content/drive/MyDrive/myllia_v23'

for d in [BASE_DIR, DATA_DIR, OUTPUT_DIR, V23_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Base: {BASE_DIR}')
print(f'V23 checkpoints: {V23_DIR}')
existing = [f for f in os.listdir(V23_DIR) if not f.startswith('.')]
if existing:
    print(f'  Found {len(existing)} existing checkpoint files:')
    for f in sorted(existing)[:20]:
        size = os.path.getsize(os.path.join(V23_DIR, f))
        print(f'    {f} ({size/1024:.0f} KB)')
else:
    print('  No existing checkpoints (fresh start)')
print('Directories ready')

In [ ]:
# ============================================================
# CELL 2: Kaggle Download (reused from V22)
# ============================================================
#@title Enter Kaggle API Token { display-mode: "form" }
KAGGLE_TOKEN = 'YOUR_KAGGLE_API_TOKEN_HERE'  #@param {type:"string"}
import shutil
if KAGGLE_TOKEN:
    os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'access_token'), 'w') as f: f.write(KAGGLE_TOKEN)
    os.chmod(os.path.join(kaggle_dir, 'access_token'), 0o600)
    !pip uninstall -y kaggle kagglehub kagglesdk 2>/dev/null
    !pip install -q --upgrade kaggle
    download_path = '/content/kaggle_data'
    os.makedirs(download_path, exist_ok=True)
    !kaggle competitions download -c echoes-of-silenced-genes -p {download_path}
    !unzip -q -o {download_path}/echoes-of-silenced-genes.zip -d {download_path}
    for f in os.listdir(download_path):
        src, dst = os.path.join(download_path, f), os.path.join(DATA_DIR, f)
        if os.path.isfile(src) and not os.path.exists(dst): shutil.copy2(src, dst)
    print(f'Data: {os.listdir(DATA_DIR)}')
else:
    print(f'Skip download. Files: {os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "None"}')

In [ ]:
# ============================================================
# CELL 3: Dependencies (V23 — adds sklearn GMM, NearestNeighbors, requests)
# ============================================================
!pip -q install scanpy anndata

import os, gc, json, math, shutil, warnings, re, datetime, time, copy
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from google.colab import files as colab_files
import requests

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Dependencies ready (V23: LOO Quality + Replogle Augmentation)')

In [ ]:
# ============================================================
# CELL 4: Config — V23 (LOO Quality + Replogle Augmentation)
#
# CASCADE NOTES:
# - Architecture: FROZEN V17 LightMLP (H=384, D=2, dropout=0.5)
# - Loss: FROZEN cosine_light (lambda=0.08, cos_right=0.0405)
# - LOO_N_RESAMPLES = 3 (V20 RUN=3 proven best LB)
# - PER_CHANNEL_LOO = False (V22 proved harmful)
# - 7 run configs: baseline + 4 LOO quality + Replogle + combined
# - Each run saves checkpoints to V23_DIR for resume
# ============================================================
USE_AMP = True
USE_TF32 = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = USE_TF32
    torch.backends.cudnn.allow_tf32 = USE_TF32
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision('high')
    except: pass

class LOOMethod(Enum):
    BASELINE = "baseline"
    NN_MATCHED = "nn_matched"
    RESIDUAL = "residual"
    ADAPTIVE = "adaptive"
    WEIGHTED = "weighted"

@dataclass
class Config:
    # Data
    NUM_GENES: int = 5127
    NUM_GENES_FULL: int = 19226
    NUM_TRAIN_PERTS: int = 80

    # PseudoBulk — 3-tier augmentation (LOCKED from V15/V20)
    N_RESAMPLES_FULL: int = 10
    N_RESAMPLES_HALF: int = 5
    HALF_CELL_FRACTION: float = 0.5
    MIN_CELLS_PER_PERT: int = 10
    MIN_CELLS_PER_CHANNEL: int = 5
    CTRL_SUBSAMPLE: int = 200
    PB_QUALITY_THRESHOLD: float = 0.3

    # LOO (LOCKED from V20 RUN=3 best LB)
    LOO_PERCENTILE: float = 5.0
    LOO_MIN_CELLS: int = 10
    LOO_N_RESAMPLES: int = 3
    LOO_SAMPLE_WEIGHT: float = 0.05
    PER_CHANNEL_LOO: bool = False  # V22 proved harmful

    # Replogle augmentation
    REPLOGLE_SAMPLE_WEIGHT: float = 0.03
    REPLOGLE_USE_ESSENTIAL: bool = True
    REPLOGLE_USE_GWPS: bool = True
    REPLOGLE_MIN_CELLS: int = 10
    REPLOGLE_GWPS_DISCOUNT: float = 0.75

    # MLP base (V6/V17 arch — LOCKED)
    EMBED_DIM: int = 64
    MLP_DEPTH: int = 2
    MLP_EPOCHS: int = 300
    MLP_LR: float = 5e-4
    EARLY_STOP: int = 50

    # V16/V17 winner HPs (LOCKED)
    MLP_HIDDEN: int = 384
    MLP_DROPOUT: float = 0.5
    WEIGHT_DECAY: float = 0.01
    COSINE_LAMBDA: float = 0.08
    COS_RIGHT: float = 0.0405
    GT_UPWEIGHT: float = 4.5
    N_MLP_ENSEMBLE: int = 30

    # Loss (LOCKED from V17)
    LOSS: str = 'cosine_light'

    # KO correction (V6 proven)
    KNOCKOUT_MULTIPLIER: float = 1.0

    # System
    N_FOLDS: int = 5
    SEED: int = 42
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(cfg.SEED)

def clean_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# === Run configurations ===
RUN_CONFIGS = [
    {"id": 0, "name": "Baseline",     "loo_method": LOOMethod.BASELINE,   "replogle": False},
    {"id": 1, "name": "NN-Matched",   "loo_method": LOOMethod.NN_MATCHED, "replogle": False},
    {"id": 2, "name": "Residual",     "loo_method": LOOMethod.RESIDUAL,   "replogle": False},
    {"id": 3, "name": "Adaptive",     "loo_method": LOOMethod.ADAPTIVE,   "replogle": False},
    {"id": 4, "name": "Weighted",     "loo_method": LOOMethod.WEIGHTED,   "replogle": False},
    {"id": 5, "name": "Replogle",     "loo_method": LOOMethod.BASELINE,   "replogle": True},
    # Run 6 added dynamically after Runs 0-5 complete
]

# === Checkpoint helpers ===
def save_checkpoint(run_id, fold_id, preds, score, w_score, wcos_score):
    """Save fold predictions and score to Drive."""
    path = os.path.join(V23_DIR, f'run{run_id}_fold{fold_id}_preds.npz')
    np.savez_compressed(path, preds=preds, score=score, w=w_score, wcos=wcos_score)
    print(f'    ✓ Checkpoint saved: run{run_id}_fold{fold_id} (score={score:.4f})')

def load_checkpoint(run_id, fold_id):
    """Load fold predictions if checkpoint exists. Returns (preds, score, w, wcos) or None."""
    path = os.path.join(V23_DIR, f'run{run_id}_fold{fold_id}_preds.npz')
    if os.path.exists(path):
        data = np.load(path)
        print(f'    ✓ Resumed: run{run_id}_fold{fold_id} (score={float(data["score"]):.4f})')
        return data['preds'], float(data['score']), float(data['w']), float(data['wcos'])
    return None

def save_run_results(run_id, run_name, cv_results):
    """Save full run results as JSON."""
    path = os.path.join(V23_DIR, f'run{run_id}_results.json')
    with open(path, 'w') as f:
        json.dump(cv_results, f, indent=2)
    print(f'  ✓ Run {run_id} results saved')

def load_run_results(run_id):
    """Load run results if they exist."""
    path = os.path.join(V23_DIR, f'run{run_id}_results.json')
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f'V23 Config: device={cfg.DEVICE}')
print(f'  Architecture: V17 LightMLP H={cfg.MLP_HIDDEN}, D={cfg.MLP_DEPTH} (FROZEN)')
print(f'  Loss: cosine_light lambda={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT} (FROZEN)')
print(f'  LOO: ALL {cfg.NUM_GENES} genes, {cfg.LOO_N_RESAMPLES} resamples, weight={cfg.LOO_SAMPLE_WEIGHT}')
print(f'  Per-Channel LOO: DISABLED (V22 proved harmful)')
print(f'  Replogle weight: {cfg.REPLOGLE_SAMPLE_WEIGHT}')
print(f'  Runs configured: {len(RUN_CONFIGS)}')
for rc in RUN_CONFIGS:
    print(f'    Run {rc["id"]}: {rc["name"]} (LOO={rc["loo_method"].value}, Replogle={rc["replogle"]})')

In [ ]:
# ============================================================
# CELL 5: Load Competition Data (from V22 — unchanged)
#
# CASCADE NOTES:
# - Loads training_data_means.csv, ground_truth_table, sample_submission
# - Builds: gene_names, control_expr, pert_names, deltas, weights_array
# - Maps validation perturbation IDs to gene indices
# ============================================================
data_path = DATA_DIR if os.path.exists(DATA_DIR) and os.listdir(DATA_DIR) else '/content/kaggle_data'
print(f'Data path: {data_path}')

train_means = pd.read_csv(os.path.join(data_path, 'training_data_means.csv'))
first_col = train_means.iloc[:, 0].astype(str)
control_mask = first_col.str.lower().str.contains('non-targeting|control', na=False)
control_row = train_means[control_mask].iloc[0] if control_mask.any() else train_means.iloc[-1]
pert_df = train_means[~control_mask] if control_mask.any() else train_means.iloc[:-1]

gene_names = list(train_means.columns[1:])
control_expr = control_row.iloc[1:].values.astype(np.float32)
pert_names = pert_df.iloc[:, 0].tolist()
pert_expr = pert_df.iloc[:, 1:].values.astype(np.float32)
deltas = pert_expr - control_expr
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
pert_to_idx = {p: i for i, p in enumerate(pert_names)}
pert_gene_indices = np.array([gene_to_idx.get(p, -1) for p in pert_names])

gt_df = pd.read_csv(os.path.join(data_path, 'training_data_ground_truth_table.csv'))
id_col = None
for c in ['pert_symbol', 'pert', 'gene', 'target_gene']:
    if c in gt_df.columns:
        id_col = c; break
if id_col:
    gt_df[id_col] = gt_df[id_col].astype(str)
    if set(pert_names).issubset(set(gt_df[id_col])):
        gt_df = gt_df.set_index(id_col).reindex(pert_names).reset_index()

weight_cols = [c for c in gt_df.columns if c.startswith('w_')]
weights_array = gt_df[weight_cols].values.astype(np.float32) if weight_cols else np.abs(deltas) + 0.1

sample_submission = pd.read_csv(os.path.join(data_path, 'sample_submission.csv'))
SUBMISSION_COLUMNS = list(sample_submission.columns)
val_pert_ids = sample_submission['pert_id'].tolist()
val_perts_df = pd.read_csv(os.path.join(data_path, 'pert_ids_val.csv'))

def _norm_pert_id(value):
    if value is None or (isinstance(value, float) and np.isnan(value)): return None
    s = str(value).strip()
    if not s: return None
    if s.startswith('pert_'): return s
    if s.isdigit(): return f'pert_{int(s)}'
    m = re.search(r'(\d+)', s)
    if m: return f'pert_{int(m.group(1))}'
    return s

val_perts_df['pert_id_norm'] = val_perts_df['pert_id'].apply(_norm_pert_id)
_sub_pert_id_norm = [_norm_pert_id(pid) for pid in val_pert_ids]

val_perts_known = val_perts_df
if 'class' in val_perts_df.columns:
    val_perts_known = val_perts_df[val_perts_df['class'].astype(str).str.lower().str.contains('val')]
pert_id_to_symbol = dict(zip(val_perts_known['pert_id_norm'], val_perts_known['pert']))
val_pert_symbols = [pert_id_to_symbol.get(pid) for pid in _sub_pert_id_norm]

gene_names_upper = {g.upper(): g for g in gene_names}
val_gene_indices = []
for symbol in val_pert_symbols:
    if symbol is not None and symbol in gene_to_idx:
        val_gene_indices.append(gene_to_idx[symbol])
    elif symbol is not None and symbol.upper() in gene_names_upper:
        val_gene_indices.append(gene_to_idx[gene_names_upper[symbol.upper()]])
    else:
        val_gene_indices.append(-1)

valid_idx_count = sum(1 for i in val_gene_indices if i >= 0)
print(f'Genes: {len(gene_names)}, Train perts: {len(pert_names)}, Val perts: {len(val_pert_symbols)}')
print(f'Val gene idx valid: {valid_idx_count}/{len(val_pert_symbols)}')
print(f'Deltas shape: {deltas.shape}, range [{deltas.min():.3f}, {deltas.max():.3f}]')
print(f'Weight cols: {len(weight_cols)}, weights shape: {weights_array.shape}')

In [ ]:
# ============================================================
# CELL 6: Competition Metric + KO Stats (from V22 — unchanged)
# ============================================================
baseline_pred = np.mean(deltas, axis=0)

baseline_wmae = None
if 'baseline_wmae' in gt_df.columns:
    baseline_wmae = gt_df['baseline_wmae'].values.astype(np.float64)
if baseline_wmae is None:
    baseline_wmae = np.mean(np.abs(deltas - baseline_pred) * weights_array, axis=1)

class CompetitionMetric:
    def __init__(self, eps=1e-12, max_log2=5.0, left=0.0, right=0.2):
        self.eps, self.max_log2, self.left, self.right = eps, max_log2, left, right

    def smoothstep(self, t):
        return t * t * (3.0 - 2.0 * t)

    def gate_smoothstep(self, x):
        t = np.clip((x - self.left) / (self.right - self.left), 0.0, 1.0)
        return self.smoothstep(t)

    def weighted_cosine(self, a, b):
        a, b = np.asarray(a, dtype=np.float64).ravel(), np.asarray(b, dtype=np.float64).ravel()
        x = np.maximum(np.abs(a), np.abs(b))
        w2 = self.gate_smoothstep(x) ** 2
        num = np.sum(w2 * a * b)
        den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
        return float(num / den) if den > self.eps else 0.0

    def score(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        if base.shape[0] != y_true.shape[0]:
            base = np.mean(np.abs(y_true - baseline_pred) * w, axis=1)
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        sum_wmae = float(np.sum(terms))
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return round(float(sum_wmae * max(0.0, wcos)), 5), sum_wmae, wcos

    def score_per_pert(self, preds, targets, weights, bl_wmae):
        y_true = np.asarray(targets, dtype=np.float64)
        y_pred = np.asarray(preds, dtype=np.float64)
        w = np.asarray(weights, dtype=np.float64)
        base = np.asarray(bl_wmae, dtype=np.float64).ravel()
        pred_wmae = np.maximum(np.mean(np.abs(y_true - y_pred) * w, axis=1), self.eps)
        base = np.maximum(base, self.eps)
        terms = np.minimum(np.log2(base / pred_wmae), self.max_log2)
        wcos = self.weighted_cosine(y_pred.ravel(), y_true.ravel())
        return terms, wcos

metric = CompetitionMetric()

ko_effects = [deltas[i, idx] for i, idx in enumerate(pert_gene_indices) if idx >= 0]
knockout_stats = {'mean': np.mean(ko_effects), 'std': np.std(ko_effects)}
print(f'baseline_wmae: mean={baseline_wmae.mean():.4f}, std={baseline_wmae.std():.4f}')
print(f'Knockout mean: {knockout_stats["mean"]:.4f}, std: {knockout_stats["std"]:.4f}')

def apply_ko(pred, idx, stats, mult=1.0):
    pred = pred.copy()
    if 0 <= idx < len(pred):
        pred[idx] = stats['mean'] * mult
    return pred

bl_score, bl_w, bl_wcos = metric.score(
    np.tile(baseline_pred, (len(pert_names), 1)), deltas, weights_array, baseline_wmae)
print(f'Baseline (mean) score: {bl_score:.4f} (W={bl_w:.4f}, wcos={bl_wcos:.4f}) -- expected ~0')

In [ ]:
# ============================================================
# CELL 7: 3-Tier PseudoBulk Generation (from V22 — unchanged)
#
# CASCADE NOTES:
# - Tier 1: Full bootstrap (10 resamples per pert, all cells)
# - Tier 2: Per-channel means (1 sample per pert×channel)
# - Tier 3: Half-cell bootstrap (5 resamples, 50% cells)
# - Quality gate: corr > 0.3 with GT delta
# - Caches ctrl_dense for LOO generation in Cell 8
# ============================================================
import scanpy as sc

print('=' * 70)
print('CELL 7: 3-Tier PseudoBulk Generation')
print('=' * 70)

h5ad_path = os.path.join(data_path, 'training_cells.h5ad')
assert os.path.exists(h5ad_path), f'training_cells.h5ad not found at {h5ad_path}'

adata_train = sc.read_h5ad(h5ad_path)
print(f'Shape: {adata_train.shape[0]} cells x {adata_train.shape[1]} genes')

X_check = adata_train.X[:200]
if sp.issparse(X_check): X_check = X_check.toarray()
data_max = X_check.max()
IS_RAW = data_max > 50
print(f'Data max (first 200 cells): {data_max:.1f} -> {"Raw UMI" if IS_RAW else "Already normalized"}')

h5_genes = list(adata_train.var_names)
h5_gene_upper = {g.upper(): i for i, g in enumerate(h5_genes)}
gene_subset_idx = []
comp_gene_order = []
for ci, g in enumerate(gene_names):
    gu = g.upper()
    if gu in h5_gene_upper:
        gene_subset_idx.append(h5_gene_upper[gu])
        comp_gene_order.append(ci)
gene_subset_idx = np.array(gene_subset_idx)
comp_gene_order = np.array(comp_gene_order)
print(f'Gene overlap: {len(gene_subset_idx)}/{cfg.NUM_GENES} ({100*len(gene_subset_idx)/cfg.NUM_GENES:.1f}%)')

CANDIDATE_COLS = ['pert_symbol', 'perturbation', 'pert', 'gene', 'target_gene',
                  'perturbation_name', 'gene_symbol', 'gene_name', 'symbol']

def _count_pert_matches(col_series, pert_names_list):
    col_upper = set(col_series.astype(str).str.upper().unique())
    return sum(1 for p in pert_names_list if p.upper() in col_upper)

pert_col_h5 = None
pert_col_matches = 0
_sgrna_to_gene = {}

for c in CANDIDATE_COLS:
    if c in adata_train.obs.columns:
        n_matches = _count_pert_matches(adata_train.obs[c], pert_names)
        print(f'  Column "{c}": {n_matches}/{len(pert_names)} matches')
        if n_matches > pert_col_matches:
            pert_col_h5, pert_col_matches = c, n_matches

if pert_col_matches < len(pert_names) // 2:
    for c in adata_train.obs.columns:
        if c == pert_col_h5: continue
        col = adata_train.obs[c]
        if col.dtype == object or str(col.dtype) == 'category':
            if 5 < col.nunique() < 50000:
                n_matches = _count_pert_matches(col, pert_names)
                if n_matches > pert_col_matches:
                    print(f'  Better column "{c}": {n_matches}/{len(pert_names)} matches')
                    pert_col_h5, pert_col_matches = c, n_matches

if pert_col_matches < len(pert_names) // 2:
    print(f'  Trying substring extraction...')
    pert_names_upper = {p.upper(): p for p in pert_names}
    for c in adata_train.obs.columns:
        col = adata_train.obs[c]
        if col.dtype != object and str(col.dtype) != 'category': continue
        extraction_map = {}
        for val in col.astype(str).unique():
            for delim in ['_', '-', '.', '|', ':']:
                for part in val.upper().split(delim):
                    part = part.strip()
                    if part in pert_names_upper:
                        extraction_map[val] = pert_names_upper[part]
                        break
                if val in extraction_map: break
        if len(extraction_map) > pert_col_matches:
            print(f'  Column "{c}": extracted {len(extraction_map)} gene names')
            pert_col_h5, pert_col_matches = c, len(extraction_map)
            _sgrna_to_gene = extraction_map

assert pert_col_h5 is not None, f'No perturbation column found!'
print(f'\nPerturbation column: "{pert_col_h5}" ({pert_col_matches}/{len(pert_names)} matches)')

pert_str_h5 = adata_train.obs[pert_col_h5].astype(str)
ctrl_mask_h5 = pert_str_h5.str.lower().str.contains('non.targeting|control|ctrl', regex=True, na=False)
if 'is_control' in adata_train.obs.columns:
    ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['is_control'].astype(bool)
if 'control' in adata_train.obs.columns and adata_train.obs['control'].dtype == bool:
    ctrl_mask_h5 = ctrl_mask_h5 | adata_train.obs['control']
ctrl_cell_idx = np.where(ctrl_mask_h5.values)[0]
print(f'Control cells: {len(ctrl_cell_idx)}')

channel_col = None
for c in ['channel', 'batch', 'lane', 'pool', 'sample']:
    if c in adata_train.obs.columns:
        n_unique = adata_train.obs[c].nunique()
        if 2 <= n_unique <= 20:
            channel_col = c; break
if channel_col is None:
    for c in adata_train.obs.columns:
        col = adata_train.obs[c]
        if col.dtype == object or str(col.dtype) == 'category':
            if 2 <= col.nunique() <= 20 and c not in [pert_col_h5]:
                channel_col = c; break

channel_names = []
if channel_col:
    channel_names = sorted(adata_train.obs[channel_col].astype(str).unique())
    print(f'Channel column: "{channel_col}" ({len(channel_names)} channels)')
else:
    print(f'WARNING: No channel column found. Tier 2 SKIPPED.')

cells_per_pert = {}
if _sgrna_to_gene:
    gene_to_sgrna_ids = {}
    for sgrna_val, gene_name in _sgrna_to_gene.items():
        gene_to_sgrna_ids.setdefault(gene_name, []).append(sgrna_val)
    for p in pert_names:
        if p in gene_to_sgrna_ids:
            all_idx = []
            for sgrna_val in gene_to_sgrna_ids[p]:
                mask = (pert_str_h5 == sgrna_val) & (~ctrl_mask_h5)
                all_idx.extend(np.where(mask.values)[0].tolist())
            if len(all_idx) >= 3:
                cells_per_pert[p] = np.array(all_idx)

for p in pert_names:
    if p in cells_per_pert: continue
    mask = (pert_str_h5.str.upper() == p.upper()) & (~ctrl_mask_h5)
    cell_idx = np.where(mask.values)[0]
    if len(cell_idx) >= 3:
        cells_per_pert[p] = cell_idx

print(f'Perturbations with cells: {len(cells_per_pert)}/{len(pert_names)}')

cells_per_pert_channel = {}
ctrl_cells_per_channel = {}
if channel_col:
    obs_ch = adata_train.obs[channel_col].astype(str).values
    for ch in channel_names:
        ch_mask = obs_ch == ch
        ctrl_ch_idx = np.where(ch_mask & ctrl_mask_h5.values)[0]
        if len(ctrl_ch_idx) > 0:
            ctrl_cells_per_channel[ch] = ctrl_ch_idx
    for p, cell_idx in cells_per_pert.items():
        for ch in channel_names:
            ch_mask_p = obs_ch[cell_idx] == ch
            ch_idx = cell_idx[ch_mask_p]
            if len(ch_idx) >= cfg.MIN_CELLS_PER_CHANNEL:
                cells_per_pert_channel[(p, ch)] = ch_idx

print(f'\nNormalizing and caching...')
t0_cache = time.time()

def normalize_cells_to_dense(X_raw, gene_subset_idx, is_raw=True):
    if sp.issparse(X_raw): X_raw = X_raw.toarray()
    X_raw = X_raw.astype(np.float64)
    if is_raw:
        totals = X_raw.sum(axis=1, keepdims=True)
        totals[totals == 0] = 1
        normalized_full = np.log2(1.0 + X_raw / totals * 10000.0)
    else:
        normalized_full = X_raw
    return normalized_full[:, gene_subset_idx].astype(np.float32)

ctrl_dense = normalize_cells_to_dense(adata_train.X[ctrl_cell_idx], gene_subset_idx, is_raw=IS_RAW)
ctrl_dense_per_channel = {}
if channel_col:
    for ch, ch_idx in ctrl_cells_per_channel.items():
        ctrl_dense_per_channel[ch] = normalize_cells_to_dense(
            adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

pert_cells_dense = {}
for p, cell_idx in tqdm(cells_per_pert.items(), desc='Caching pert cells'):
    pert_cells_dense[p] = normalize_cells_to_dense(
        adata_train.X[cell_idx], gene_subset_idx, is_raw=IS_RAW)

pert_cells_dense_channel = {}
if channel_col:
    for (p, ch), ch_idx in tqdm(cells_per_pert_channel.items(), desc='Caching pert-channel'):
        pert_cells_dense_channel[(p, ch)] = normalize_cells_to_dense(
            adata_train.X[ch_idx], gene_subset_idx, is_raw=IS_RAW)

print(f'  Caching done in {time.time()-t0_cache:.1f}s')
del adata_train; clean_memory()

print(f'\nGenerating 3-Tier PseudoBulk...')
rng = np.random.RandomState(cfg.SEED)
n_ctrl = ctrl_dense.shape[0]
n_ctrl_sub = min(n_ctrl, cfg.CTRL_SUBSAMPLE)

pb_deltas_list, pb_pert_indices_list, pb_pert_names_list = [], [], []
pb_tiers_list, pb_n_cells_list = [], []

# Tier 1: Full Bootstrap
n_t1 = 0
for p in pert_names:
    if p not in pert_cells_dense: continue
    pert_cells = pert_cells_dense[p]
    n_pert = pert_cells.shape[0]
    if n_pert < cfg.MIN_CELLS_PER_PERT: continue
    gi = pert_gene_indices[pert_to_idx[p]]
    for _ in range(cfg.N_RESAMPLES_FULL):
        pert_idx = rng.randint(0, n_pert, size=n_pert)
        ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
        delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
        pb_deltas_list.append(delta)
        pb_pert_indices_list.append(gi)
        pb_pert_names_list.append(p)
        pb_tiers_list.append(1); pb_n_cells_list.append(n_pert); n_t1 += 1
print(f'  Tier 1: {n_t1}')

# Tier 2: Per-Channel
n_t2 = 0
if channel_col and ctrl_dense_per_channel:
    for p in pert_names:
        gi = pert_gene_indices[pert_to_idx[p]]
        for ch in channel_names:
            if (p, ch) not in pert_cells_dense_channel: continue
            if ch not in ctrl_dense_per_channel: continue
            pert_ch = pert_cells_dense_channel[(p, ch)]
            ctrl_ch = ctrl_dense_per_channel[ch]
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = pert_ch.mean(axis=0) - ctrl_ch.mean(axis=0)
            pb_deltas_list.append(delta)
            pb_pert_indices_list.append(gi)
            pb_pert_names_list.append(p)
            pb_tiers_list.append(2); pb_n_cells_list.append(pert_ch.shape[0]); n_t2 += 1
print(f'  Tier 2: {n_t2}')

# Tier 3: Half-Cell
n_t3 = 0
for p in pert_names:
    if p not in pert_cells_dense: continue
    pert_cells = pert_cells_dense[p]
    n_pert = pert_cells.shape[0]
    if n_pert < cfg.MIN_CELLS_PER_PERT: continue
    gi = pert_gene_indices[pert_to_idx[p]]
    half_n = max(3, int(n_pert * cfg.HALF_CELL_FRACTION))
    for _ in range(cfg.N_RESAMPLES_HALF):
        pert_idx = rng.choice(n_pert, size=half_n, replace=False)
        ctrl_idx = rng.randint(0, n_ctrl, size=n_ctrl_sub)
        delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        delta[comp_gene_order] = pert_cells[pert_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
        pb_deltas_list.append(delta)
        pb_pert_indices_list.append(gi)
        pb_pert_names_list.append(p)
        pb_tiers_list.append(3); pb_n_cells_list.append(half_n); n_t3 += 1
print(f'  Tier 3: {n_t3}')

# Quality Gate
pb_deltas_raw = np.array(pb_deltas_list, dtype=np.float32)
pb_pert_indices_raw = np.array(pb_pert_indices_list)
pb_pert_names_raw = pb_pert_names_list
pb_tiers_raw = np.array(pb_tiers_list)
pb_n_cells_raw = np.array(pb_n_cells_list)

keep_mask = np.ones(len(pb_deltas_raw), dtype=bool)
for i in range(len(pb_deltas_raw)):
    pn = pb_pert_names_raw[i]
    corr = np.corrcoef(pb_deltas_raw[i], deltas[pert_to_idx[pn]])[0, 1]
    if np.isnan(corr) or corr < cfg.PB_QUALITY_THRESHOLD:
        keep_mask[i] = False

print(f'  Quality gate: kept {keep_mask.sum()}/{len(pb_deltas_raw)}')

pb_deltas = pb_deltas_raw[keep_mask]
pb_pert_indices = pb_pert_indices_raw[keep_mask]
pb_pert_names = [pb_pert_names_raw[i] for i in range(len(pb_pert_names_raw)) if keep_mask[i]]
pb_tiers = pb_tiers_raw[keep_mask]
pb_n_cells = pb_n_cells_raw[keep_mask]

pb_cell_weights = np.sqrt(pb_n_cells.astype(np.float32))
mean_sqrt = pb_cell_weights.mean()
if mean_sqrt > 0: pb_cell_weights /= mean_sqrt

pb_weights = np.zeros((len(pb_deltas), cfg.NUM_GENES), dtype=np.float32)
for i, pn in enumerate(pb_pert_names):
    pb_weights[i] = weights_array[pert_to_idx[pn]]

del pert_cells_dense, pert_cells_dense_channel
clean_memory()

n_pb_perts = len(set(pb_pert_names))
print(f'\nPB ready: {n_pb_perts} perts, {len(pb_deltas)} samples')
print(f'Total so far: {len(pb_deltas)} PB + {len(pert_names)} GT = {len(pb_deltas) + len(pert_names)}')

In [ ]:
# ============================================================
# CELL 8: LOO Quality Methods — All 5 Methods with Diagnostics
#
# CASCADE NOTES:
# - BASELINE: Bottom 5th percentile by raw expression (V20 RUN=3)
# - NN_MATCHED: Nearest-neighbor matched pairs in PCA space
# - RESIDUAL: Regression residual selection (low GIVEN cell state)
# - ADAPTIVE: GMM bimodal detection per gene
# - WEIGHTED: Continuous expression weighting (no hard cutoff)
# - Each method returns (gene_indices, deltas, sample_weights) arrays
# - LOO is cached per method to V23_DIR for resume
# ============================================================
print('=' * 70)
print('CELL 8: LOO Quality Methods')
print('=' * 70)

# Map competition gene index → column in ctrl_dense
comp_gene_to_col = {}
for col_j, comp_gi in enumerate(comp_gene_order):
    comp_gene_to_col[comp_gi] = col_j
n_ctrl_cells = ctrl_dense.shape[0]
n_overlap_genes = ctrl_dense.shape[1]
percentile_threshold = cfg.LOO_PERCENTILE

print(f'Control cells: {n_ctrl_cells}, Overlap genes: {n_overlap_genes}')
print(f'LOO resamples: {cfg.LOO_N_RESAMPLES}, Percentile: {percentile_threshold}%')

# === Precompute PCA for NN-Matched and Residual methods ===
print('\nPrecomputing PCA on control cells (50 components)...')
t0_pca = time.time()
pca_model = PCA(n_components=min(50, n_ctrl_cells - 1, n_overlap_genes), random_state=cfg.SEED)
ctrl_pca = pca_model.fit_transform(ctrl_dense)  # (n_ctrl, 50)
print(f'  PCA done in {time.time()-t0_pca:.1f}s, shape={ctrl_pca.shape}, var_explained={pca_model.explained_variance_ratio_.sum():.3f}')

# Precompute PCA for residual method (20 components)
pca_20 = PCA(n_components=min(20, n_ctrl_cells - 1, n_overlap_genes), random_state=cfg.SEED)
ctrl_pca_20 = pca_20.fit_transform(ctrl_dense)  # (n_ctrl, 20)

# === LOO Generation Functions ===

def _loo_baseline(rng):
    """Baseline: bottom percentile by raw expression."""
    loo_deltas, loo_gis = [], []
    n_genes_done = 0
    for comp_gi in range(cfg.NUM_GENES):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]
        threshold = np.percentile(gene_expr, percentile_threshold)
        low_mask = gene_expr <= threshold
        n_low = low_mask.sum()
        if n_low < cfg.LOO_MIN_CELLS: continue
        low_cells = ctrl_dense[low_mask]
        for _ in range(cfg.LOO_N_RESAMPLES):
            boot_idx = rng.randint(0, n_low, size=n_low)
            ctrl_idx = rng.randint(0, n_ctrl_cells, size=min(n_ctrl_cells, cfg.CTRL_SUBSAMPLE))
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = low_cells[boot_idx].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
            loo_deltas.append(delta)
            loo_gis.append(comp_gi)
        n_genes_done += 1
    return np.array(loo_gis, dtype=np.int64), np.array(loo_deltas, dtype=np.float32), n_genes_done


def _loo_nn_matched(rng):
    """NN-Matched: for each low-G cell, find nearest normal-G neighbor in PCA space."""
    loo_deltas, loo_gis = [], []
    n_genes_done = 0
    for comp_gi in range(cfg.NUM_GENES):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]
        threshold = np.percentile(gene_expr, percentile_threshold)
        low_mask = gene_expr <= threshold
        n_low = low_mask.sum()
        if n_low < cfg.LOO_MIN_CELLS: continue

        high_mask = ~low_mask
        if high_mask.sum() < 5: continue

        # Exclude gene G's contribution from PCA (approximate)
        pca_component_g = pca_model.components_[:, col_j:col_j+1]  # (50, 1)
        gene_g_centered = ctrl_dense[:, col_j:col_j+1] - ctrl_dense[:, col_j].mean()
        correction = gene_g_centered @ pca_component_g.T  # (n_ctrl, 50)
        ctrl_pca_no_g = ctrl_pca - correction

        low_pca = ctrl_pca_no_g[low_mask]
        high_pca = ctrl_pca_no_g[high_mask]
        high_cells = ctrl_dense[high_mask]

        nn = NearestNeighbors(n_neighbors=1, algorithm='auto')
        nn.fit(high_pca)

        for _ in range(cfg.LOO_N_RESAMPLES):
            boot_low_idx = rng.randint(0, n_low, size=n_low)
            boot_low_pca = low_pca[boot_low_idx]
            _, match_idx = nn.kneighbors(boot_low_pca)
            match_idx = match_idx.ravel()

            low_boot_cells = ctrl_dense[low_mask][boot_low_idx]
            matched_high_cells = high_cells[match_idx]
            pair_deltas = low_boot_cells - matched_high_cells
            avg_delta = pair_deltas.mean(axis=0)

            full_delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            full_delta[comp_gene_order] = avg_delta
            loo_deltas.append(full_delta)
            loo_gis.append(comp_gi)
        n_genes_done += 1
    return np.array(loo_gis, dtype=np.int64), np.array(loo_deltas, dtype=np.float32), n_genes_done


def _loo_residual(rng):
    """Residual: select cells where gene G is lower than expected given cell state."""
    loo_deltas, loo_gis = [], []
    n_genes_done = 0
    for comp_gi in range(cfg.NUM_GENES):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]

        # Linear regression: gene_expr ~ PC1 + ... + PC20
        X_reg = np.column_stack([np.ones(n_ctrl_cells), ctrl_pca_20])
        try:
            beta, _, _, _ = np.linalg.lstsq(X_reg, gene_expr, rcond=None)
        except:
            continue
        predicted = X_reg @ beta
        residuals = gene_expr - predicted

        # Select cells with most negative residuals (bottom 5%)
        residual_threshold = np.percentile(residuals, percentile_threshold)
        low_mask = residuals <= residual_threshold
        n_low = low_mask.sum()
        if n_low < cfg.LOO_MIN_CELLS: continue

        # Baseline: cells with residuals near 0 (middle 50%)
        mid_mask = (residuals >= np.percentile(residuals, 25)) & (residuals <= np.percentile(residuals, 75))
        if mid_mask.sum() < 10:
            mid_mask = ~low_mask

        low_cells = ctrl_dense[low_mask]
        mid_cells = ctrl_dense[mid_mask]
        n_mid = mid_cells.shape[0]

        for _ in range(cfg.LOO_N_RESAMPLES):
            boot_low = rng.randint(0, n_low, size=n_low)
            boot_mid = rng.randint(0, n_mid, size=min(n_mid, cfg.CTRL_SUBSAMPLE))
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = low_cells[boot_low].mean(axis=0) - mid_cells[boot_mid].mean(axis=0)
            loo_deltas.append(delta)
            loo_gis.append(comp_gi)
        n_genes_done += 1
    return np.array(loo_gis, dtype=np.int64), np.array(loo_deltas, dtype=np.float32), n_genes_done


def _loo_adaptive(rng):
    """Adaptive: GMM bimodal detection per gene."""
    loo_deltas, loo_gis = [], []
    n_genes_done = 0
    n_bimodal = 0
    for comp_gi in range(cfg.NUM_GENES):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]

        # Fit 2-component GMM
        try:
            gmm = GaussianMixture(n_components=2, random_state=cfg.SEED, max_iter=50)
            gmm.fit(gene_expr.reshape(-1, 1))
            means = gmm.means_.ravel()
            stds = np.sqrt(gmm.covariances_.ravel())
            separation = abs(means[0] - means[1]) / max(stds.max(), 1e-6)
        except:
            separation = 0

        if separation > 1.5:
            # BIMODAL: select all cells in lower component
            labels = gmm.predict(gene_expr.reshape(-1, 1))
            lower_comp = 0 if means[0] < means[1] else 1
            low_mask = labels == lower_comp
            n_bimodal += 1
        else:
            # UNIMODAL: use stricter 2nd percentile
            threshold = np.percentile(gene_expr, 2.0)
            low_mask = gene_expr <= threshold

        n_low = low_mask.sum()
        if n_low < cfg.LOO_MIN_CELLS: continue

        low_cells = ctrl_dense[low_mask]
        for _ in range(cfg.LOO_N_RESAMPLES):
            boot_low = rng.randint(0, n_low, size=n_low)
            ctrl_idx = rng.randint(0, n_ctrl_cells, size=min(n_ctrl_cells, cfg.CTRL_SUBSAMPLE))
            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = low_cells[boot_low].mean(axis=0) - ctrl_dense[ctrl_idx].mean(axis=0)
            loo_deltas.append(delta)
            loo_gis.append(comp_gi)
        n_genes_done += 1

    print(f'    Adaptive: {n_bimodal} bimodal / {n_genes_done} total genes')
    return np.array(loo_gis, dtype=np.int64), np.array(loo_deltas, dtype=np.float32), n_genes_done


def _loo_weighted(rng):
    """Weighted: continuous expression weighting, no hard cutoff."""
    loo_deltas, loo_gis = [], []
    n_genes_done = 0
    for comp_gi in range(cfg.NUM_GENES):
        if comp_gi not in comp_gene_to_col: continue
        col_j = comp_gene_to_col[comp_gi]
        gene_expr = ctrl_dense[:, col_j]
        median_expr = np.median(gene_expr)

        # Weight = max(0, median - expr) / (median + eps)
        raw_weights = np.maximum(0, median_expr - gene_expr) / (median_expr + 1e-6)
        if raw_weights.sum() < 1e-8: continue
        n_positive = (raw_weights > 0).sum()
        if n_positive < 5: continue

        weights_norm = raw_weights / raw_weights.sum()

        # Baseline: cells above median
        above_mask = gene_expr > median_expr
        if above_mask.sum() < 5: continue
        above_cells = ctrl_dense[above_mask]
        n_above = above_cells.shape[0]

        for _ in range(cfg.LOO_N_RESAMPLES):
            # Dirichlet perturbation of weights for diversity
            dirichlet_noise = rng.dirichlet(10.0 * np.ones(n_ctrl_cells))
            w_resampled = weights_norm * dirichlet_noise
            w_resampled = w_resampled / (w_resampled.sum() + 1e-12)

            weighted_low_mean = (ctrl_dense * w_resampled[:, np.newaxis]).sum(axis=0)
            above_idx = rng.randint(0, n_above, size=min(n_above, cfg.CTRL_SUBSAMPLE))
            above_mean = above_cells[above_idx].mean(axis=0)

            delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
            delta[comp_gene_order] = weighted_low_mean - above_mean
            loo_deltas.append(delta)
            loo_gis.append(comp_gi)
        n_genes_done += 1
    return np.array(loo_gis, dtype=np.int64), np.array(loo_deltas, dtype=np.float32), n_genes_done


# === Master LOO generator with caching ===
def generate_loo_samples(method: LOOMethod, seed=42):
    """Generate LOO samples using specified method. Caches to Drive."""
    cache_path = os.path.join(V23_DIR, f'loo_{method.value}.npz')
    if os.path.exists(cache_path):
        data = np.load(cache_path)
        print(f'  ✓ Loaded cached LOO ({method.value}): {len(data["gis"])} samples')
        return data['gis'], data['deltas'], data['n_genes']

    print(f'  Generating LOO ({method.value})...')
    t0 = time.time()
    rng = np.random.RandomState(seed)

    if method == LOOMethod.BASELINE:
        gis, delts, n_g = _loo_baseline(rng)
    elif method == LOOMethod.NN_MATCHED:
        gis, delts, n_g = _loo_nn_matched(rng)
    elif method == LOOMethod.RESIDUAL:
        gis, delts, n_g = _loo_residual(rng)
    elif method == LOOMethod.ADAPTIVE:
        gis, delts, n_g = _loo_adaptive(rng)
    elif method == LOOMethod.WEIGHTED:
        gis, delts, n_g = _loo_weighted(rng)
    else:
        raise ValueError(f'Unknown method: {method}')

    elapsed = time.time() - t0
    print(f'    {method.value}: {n_g} genes, {len(gis)} samples in {elapsed:.1f}s')

    # Cache to Drive
    np.savez_compressed(cache_path, gis=gis, deltas=delts, n_genes=n_g)
    print(f'    ✓ Cached to {cache_path}')

    return gis, delts, n_g

# === Generate all LOO variants now (so they're cached for the multi-run engine) ===
print('\n--- Pre-generating all LOO variants ---')
loo_cache = {}
for method in LOOMethod:
    gis, delts, n_g = generate_loo_samples(method, seed=cfg.SEED)
    loo_cache[method] = (gis, delts)
    print(f'  {method.value}: {n_g} genes → {len(gis)} samples, mean|delta|={np.abs(delts).mean():.6f}')

# === Diagnostic Charts ===
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('V23 LOO Quality Methods Comparison', fontsize=14, fontweight='bold')

method_colors = {
    LOOMethod.BASELINE: '#2196F3', LOOMethod.NN_MATCHED: '#4CAF50',
    LOOMethod.RESIDUAL: '#FF9800', LOOMethod.ADAPTIVE: '#E91E63',
    LOOMethod.WEIGHTED: '#9C27B0'
}

# Chart 1: Sample count per method
ax = axes[0, 0]
method_names = [m.value for m in LOOMethod]
sample_counts = [len(loo_cache[m][0]) for m in LOOMethod]
bars = ax.bar(method_names, sample_counts, color=[method_colors[m] for m in LOOMethod])
ax.set_title('LOO Sample Count per Method')
ax.set_ylabel('# Samples')
for bar, cnt in zip(bars, sample_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{cnt:,}', ha='center', va='bottom', fontsize=8)

# Chart 2: Mean |delta| distribution per method
ax = axes[0, 1]
for m in LOOMethod:
    delts = loo_cache[m][1]
    abs_means = np.abs(delts).mean(axis=1)
    ax.hist(abs_means, bins=60, alpha=0.4, color=method_colors[m], label=m.value, density=True)
ax.set_title('LOO Mean |Delta| Distribution')
ax.set_xlabel('Mean |Delta| per Sample')
ax.legend(fontsize=7)

# Chart 3: KO delta comparison (self-targeting gene)
ax = axes[0, 2]
for m in LOOMethod:
    gis, delts = loo_cache[m]
    ko_deltas = [delts[i, gis[i]] for i in range(len(gis)) if gis[i] < cfg.NUM_GENES]
    if ko_deltas:
        ax.hist(ko_deltas, bins=60, alpha=0.4, color=method_colors[m], label=f'{m.value} ({len(ko_deltas)})', density=True)
ax.axvline(knockout_stats['mean'], color='red', linestyle='--', linewidth=2, label=f'GT KO mean={knockout_stats["mean"]:.3f}')
ax.set_title('Self-Targeting (KO) Delta')
ax.set_xlabel('KO Gene Delta')
ax.legend(fontsize=6)

# Chart 4: Correlation with GT for matched genes
ax = axes[1, 0]
gt_corrs = {}
for m in LOOMethod:
    gis, delts = loo_cache[m]
    corrs = []
    for pi, p in enumerate(pert_names):
        gi = pert_gene_indices[pi]
        if gi < 0: continue
        loo_mask = gis == gi
        if loo_mask.sum() == 0: continue
        loo_avg = delts[loo_mask].mean(axis=0)
        gt_delta = deltas[pi]
        c = np.corrcoef(loo_avg, gt_delta)[0, 1]
        if not np.isnan(c): corrs.append(c)
    gt_corrs[m] = corrs
    if corrs:
        ax.hist(corrs, bins=30, alpha=0.4, color=method_colors[m],
                label=f'{m.value} (mean={np.mean(corrs):.3f})', density=True)
ax.set_title('LOO↔GT Correlation (matched genes)')
ax.set_xlabel('Pearson Correlation')
ax.legend(fontsize=6)

# Chart 5: Summary stats table
ax = axes[1, 1]
ax.axis('off')
summary_lines = 'Method         Samples  Genes  |Delta|   KO_delta  GT_corr\n'
summary_lines += '─' * 65 + '\n'
for m in LOOMethod:
    gis, delts = loo_cache[m]
    n_genes = len(set(gis.tolist()))
    abs_mean = np.abs(delts).mean()
    ko_vals = [delts[i, gis[i]] for i in range(len(gis)) if gis[i] < cfg.NUM_GENES]
    ko_mean = np.mean(ko_vals) if ko_vals else 0
    gt_c = np.mean(gt_corrs.get(m, [0]))
    summary_lines += f'{m.value:14s} {len(gis):>6,}  {n_genes:>5}  {abs_mean:.5f}  {ko_mean:+.4f}   {gt_c:.4f}\n'
summary_lines += '─' * 65 + '\n'
summary_lines += f'GT KO mean: {knockout_stats["mean"]:.4f}\n'
ax.text(0.02, 0.95, summary_lines, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Chart 6: Gene coverage overlap
ax = axes[1, 2]
from matplotlib_venn import venn2
try:
    from matplotlib_venn import venn2
    base_genes = set(loo_cache[LOOMethod.BASELINE][0].tolist())
    nn_genes = set(loo_cache[LOOMethod.NN_MATCHED][0].tolist())
    venn2([base_genes, nn_genes], set_labels=['Baseline', 'NN-Matched'], ax=ax)
    ax.set_title('Gene Coverage: Baseline vs NN-Matched')
except ImportError:
    ax.text(0.5, 0.5, 'matplotlib-venn not installed\nAll methods cover ~same genes',
            ha='center', va='center', transform=ax.transAxes, fontsize=10)
    ax.set_title('Gene Coverage (venn unavailable)')

plt.tight_layout()
plt.show()

print('\n' + '=' * 70)
print('LOO Quality Methods ready. All variants cached to Drive.')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 9: Replogle K562 Download & Processing
#
# CASCADE NOTES:
# - Downloads K562 Essential + GWPS scPerturb h5ad files
# - **RAM-SAFE**: files > 3 GB use backed (memory-mapped) mode
#   with single-pass streaming — never loads full matrix into RAM
# - Smaller files use chunked mean (also RAM-safe)
# - Maps Replogle genes → competition 5127-gene space
# - Cached to V23_DIR for resume
# ============================================================
print('=' * 70)
print('CELL 9: Replogle K562 Data Augmentation')
print('=' * 70)

REPLOGLE_CACHE = os.path.join(V23_DIR, 'replogle_augmentation.npz')

if os.path.exists(REPLOGLE_CACHE):
    print('Loading cached Replogle augmentation...')
    rdata = np.load(REPLOGLE_CACHE, allow_pickle=True)
    replogle_deltas = rdata['deltas']
    replogle_gene_indices = rdata['gene_indices']
    replogle_names = list(rdata['names'])
    n_replogle = len(replogle_deltas)
    print(f'  ✓ Loaded {n_replogle} Replogle perturbations from cache')
else:
    import scanpy as sc

    SCPERTURB_URLS = {
        'ReplogleK562essential': 'https://zenodo.org/records/13350497/files/ReplogleWeissman2022_K562_essential.h5ad?download=1',
        'ReplogleK562gwps': 'https://zenodo.org/records/13350497/files/ReplogleWeissman2022_K562_gwps.h5ad?download=1',
    }

    def download_h5ad(name, url, dest_dir='/content'):
        """Download scPerturb h5ad file with progress."""
        fname = f'{name}.h5ad'
        fpath = os.path.join(dest_dir, fname)
        if os.path.exists(fpath):
            print(f'  {name}: already downloaded ({os.path.getsize(fpath)/1e9:.1f} GB)')
            return fpath
        print(f'  Downloading {name}...')
        r = requests.get(url, stream=True)
        total = int(r.headers.get('content-length', 0))
        downloaded = 0
        with open(fpath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192*1024):
                f.write(chunk)
                downloaded += len(chunk)
                if total > 0:
                    pct = 100*downloaded/total
                    print(f'\r  {name}: {downloaded/1e9:.2f}/{total/1e9:.2f} GB ({pct:.0f}%)', end='', flush=True)
        print(f'\n  ✓ {name} downloaded ({os.path.getsize(fpath)/1e9:.1f} GB)')
        return fpath

    def _normalize_chunk(X_chunk, is_raw):
        """Normalize a small dense chunk: log2(1 + UMI/total * 10000)."""
        X_chunk = X_chunk.astype(np.float64)
        if is_raw:
            totals = X_chunk.sum(axis=1, keepdims=True)
            totals[totals == 0] = 1
            return np.log2(1.0 + X_chunk / totals * 10000.0)
        return X_chunk

    def _chunked_mean(X_sparse, row_indices, is_raw, chunk_size=5000):
        """Compute mean of normalized rows WITHOUT materializing full dense matrix."""
        n_rows = len(row_indices)
        n_genes = X_sparse.shape[1]
        running_sum = np.zeros(n_genes, dtype=np.float64)
        for start in range(0, n_rows, chunk_size):
            end = min(start + chunk_size, n_rows)
            idx_chunk = row_indices[start:end]
            X_chunk = X_sparse[idx_chunk]
            if sp.issparse(X_chunk):
                X_chunk = X_chunk.toarray()
            X_norm = _normalize_chunk(X_chunk, is_raw)
            running_sum += X_norm.sum(axis=0)
            del X_chunk, X_norm
        return (running_sum / n_rows).astype(np.float32)

    def _stream_perturbation_deltas(adata_backed, pert_labels, ctrl_mask, is_raw,
                                     min_cells, chunk_size=8000):
        """Single-pass streaming: read entire backed dataset in chunks,
        accumulating running sums for control + each perturbation.
        Peak RAM: chunk_size × n_genes × 8 bytes (~500 MB for 8K × 8563)."""
        n_cells = adata_backed.shape[0]
        n_genes = adata_backed.shape[1]

        # Build lookup: which perturbations exist (non-control)
        unique_perts = sorted(set(
            p for p in pert_labels.unique()
            if not ctrl_mask[pert_labels == p].all()
        ))
        pert_sums = {p: np.zeros(n_genes, dtype=np.float64) for p in unique_perts}
        pert_counts = {p: 0 for p in unique_perts}
        ctrl_sum = np.zeros(n_genes, dtype=np.float64)
        ctrl_count = 0

        labels_arr = pert_labels.values.astype(str)
        ctrl_arr = ctrl_mask.values
        pert_set = set(unique_perts)

        n_chunks = (n_cells + chunk_size - 1) // chunk_size
        for ci in tqdm(range(n_chunks), desc='  Streaming chunks'):
            start = ci * chunk_size
            end = min(start + chunk_size, n_cells)

            # Read contiguous slice from disk (memory-mapped)
            X_chunk = adata_backed.X[start:end]
            if sp.issparse(X_chunk):
                X_chunk = X_chunk.toarray()
            X_norm = _normalize_chunk(X_chunk, is_raw)
            del X_chunk

            chunk_ctrl = ctrl_arr[start:end]
            chunk_labels = labels_arr[start:end]

            # Accumulate control cells
            if chunk_ctrl.any():
                ctrl_sum += X_norm[chunk_ctrl].sum(axis=0)
                ctrl_count += int(chunk_ctrl.sum())

            # Accumulate non-control cells per perturbation
            non_ctrl = ~chunk_ctrl
            if non_ctrl.any():
                nc_labels = chunk_labels[non_ctrl]
                nc_X = X_norm[non_ctrl]
                for p in np.unique(nc_labels):
                    if p not in pert_set:
                        continue
                    pmask = nc_labels == p
                    pert_sums[p] += nc_X[pmask].sum(axis=0)
                    pert_counts[p] += int(pmask.sum())
                del nc_labels, nc_X

            del X_norm
            if ci % 20 == 0:
                gc.collect()

        print(f'  Control cells processed: {ctrl_count}')
        ctrl_mean = (ctrl_sum / max(ctrl_count, 1)).astype(np.float32)

        deltas_dict = {}
        for p in unique_perts:
            if pert_counts[p] < min_cells:
                continue
            p_mean = (pert_sums[p] / pert_counts[p]).astype(np.float32)
            deltas_dict[p] = p_mean - ctrl_mean

        del pert_sums, ctrl_sum
        gc.collect()
        return deltas_dict

    def process_scperturb_h5ad(path, dataset_name):
        """Process scPerturb h5ad → perturbation deltas.
        Files > 3 GB use backed mode (memory-mapped) + single-pass streaming.
        Smaller files use standard load + chunked means."""
        file_size_gb = os.path.getsize(path) / 1e9
        use_backed = file_size_gb > 3.0

        print(f'\n  --- Processing {dataset_name} ({file_size_gb:.1f} GB) ---')
        if use_backed:
            print(f'  Using BACKED (memory-mapped) mode — RAM safe')
            adata = sc.read_h5ad(path, backed='r')
        else:
            adata = sc.read_h5ad(path)
        print(f'  Shape: {adata.shape}')

        # Detect raw vs normalized
        X_sample = adata.X[:100]
        if sp.issparse(X_sample):
            X_sample = X_sample.toarray()
        is_raw = X_sample.max() > 50
        print(f'  Data max: {X_sample.max():.1f} → {"Raw UMI" if is_raw else "Normalized"}')
        del X_sample

        sc_genes = list(adata.var_names)

        # Find perturbation column
        pert_col = None
        for c in ['perturbation', 'gene', 'pert_symbol', 'target_gene', 'perturbation_name']:
            if c in adata.obs.columns:
                pert_col = c; break
        if pert_col is None:
            for c in adata.obs.columns:
                col = adata.obs[c]
                if (col.dtype == object or str(col.dtype) == 'category') and 50 < col.nunique() < 20000:
                    pert_col = c; break
        assert pert_col is not None, f'No perturbation column found in {dataset_name}'
        print(f'  Perturbation column: "{pert_col}" ({adata.obs[pert_col].nunique()} unique)')

        pert_labels = adata.obs[pert_col].astype(str)
        ctrl_mask = pert_labels.str.lower().str.contains('control|non.targeting', regex=True, na=False)
        if 'is_control' in adata.obs.columns:
            ctrl_mask = ctrl_mask | adata.obs['is_control'].astype(bool)
        ctrl_idx = np.where(ctrl_mask.values)[0]
        print(f'  Control cells: {len(ctrl_idx)}')

        if len(ctrl_idx) == 0:
            print(f'  WARNING: No control cells found, skipping {dataset_name}')
            if use_backed: adata.file.close()
            return {}, sc_genes

        if use_backed:
            # === SINGLE-PASS STREAMING (for large files) ===
            deltas_dict = _stream_perturbation_deltas(
                adata, pert_labels, ctrl_mask, is_raw, cfg.REPLOGLE_MIN_CELLS)
            adata.file.close()
        else:
            # === CHUNKED MEANS (for smaller files) ===
            print(f'  Computing control mean (chunked, {len(ctrl_idx)} cells)...')
            ctrl_mean = _chunked_mean(adata.X, ctrl_idx, is_raw)
            gc.collect()

            unique_perts = [p for p in pert_labels.unique() if not ctrl_mask[pert_labels == p].all()]
            deltas_dict = {}
            print(f'  Computing deltas for {len(unique_perts)} perturbations (chunked)...')
            for p in tqdm(unique_perts, desc=f'  {dataset_name} deltas'):
                p_mask = (pert_labels == p) & (~ctrl_mask)
                p_idx = np.where(p_mask.values)[0]
                if len(p_idx) < cfg.REPLOGLE_MIN_CELLS: continue
                p_mean = _chunked_mean(adata.X, p_idx, is_raw)
                deltas_dict[p] = (p_mean - ctrl_mean).astype(np.float32)
            del adata

        gc.collect()
        print(f'  {dataset_name}: {len(deltas_dict)} perturbations with >= {cfg.REPLOGLE_MIN_CELLS} cells')
        return deltas_dict, sc_genes

    # === Download and process ===
    all_replogle_deltas = {}
    all_sc_genes = None

    for name, url in SCPERTURB_URLS.items():
        if 'essential' in name.lower() and not cfg.REPLOGLE_USE_ESSENTIAL: continue
        if 'gwps' in name.lower() and not cfg.REPLOGLE_USE_GWPS: continue
        h5_path = download_h5ad(name, url)
        deltas_dict, sc_genes = process_scperturb_h5ad(h5_path, name)
        if all_sc_genes is None:
            all_sc_genes = sc_genes
        for p, d in deltas_dict.items():
            source_tag = 'ess' if 'essential' in name.lower() else 'gwps'
            all_replogle_deltas[f'{source_tag}_{p}'] = (d, source_tag)
        # Free memory + disk
        del deltas_dict
        gc.collect()
        if os.path.exists(h5_path):
            os.remove(h5_path)
            print(f'  Removed {h5_path} to free disk space')

    # === Map to competition gene space ===
    print(f'\nMapping {len(all_replogle_deltas)} Replogle perturbations to competition genes...')
    sc_gene_upper = {g.upper(): i for i, g in enumerate(all_sc_genes)}

    comp_to_replogle = {}
    for ci, g in enumerate(gene_names):
        gu = g.upper()
        if gu in sc_gene_upper:
            comp_to_replogle[ci] = sc_gene_upper[gu]

    gene_overlap = len(comp_to_replogle)
    print(f'  Gene overlap: {gene_overlap}/{cfg.NUM_GENES} ({100*gene_overlap/cfg.NUM_GENES:.1f}%)')

    replogle_deltas_list = []
    replogle_gene_indices_list = []
    replogle_names_list = []

    for rpert_name, (rpert_delta, source_tag) in all_replogle_deltas.items():
        actual_gene = rpert_name.split('_', 1)[1] if '_' in rpert_name else rpert_name
        actual_gene_upper = actual_gene.upper()

        comp_gi = -1
        if actual_gene in gene_to_idx:
            comp_gi = gene_to_idx[actual_gene]
        elif actual_gene_upper in gene_names_upper:
            comp_gi = gene_to_idx[gene_names_upper[actual_gene_upper]]

        if comp_gi < 0: continue

        mapped_delta = np.zeros(cfg.NUM_GENES, dtype=np.float32)
        for comp_ci, rep_ci in comp_to_replogle.items():
            if rep_ci < len(rpert_delta):
                mapped_delta[comp_ci] = rpert_delta[rep_ci]

        replogle_deltas_list.append(mapped_delta)
        replogle_gene_indices_list.append(comp_gi)
        replogle_names_list.append(rpert_name)

    del all_replogle_deltas; gc.collect()

    replogle_deltas = np.array(replogle_deltas_list, dtype=np.float32)
    replogle_gene_indices = np.array(replogle_gene_indices_list, dtype=np.int64)
    replogle_names = replogle_names_list
    n_replogle = len(replogle_deltas)

    np.savez_compressed(REPLOGLE_CACHE,
                        deltas=replogle_deltas,
                        gene_indices=replogle_gene_indices,
                        names=np.array(replogle_names, dtype=object))
    print(f'\n  ✓ Cached {n_replogle} Replogle perturbations to Drive')

# === Build Replogle weights and sample weights ===
avg_weights = weights_array.mean(axis=0)
replogle_weights = np.tile(avg_weights, (n_replogle, 1))

replogle_sample_weights = np.full(n_replogle, cfg.REPLOGLE_SAMPLE_WEIGHT, dtype=np.float32)
for i, name in enumerate(replogle_names):
    if name.startswith('gwps_'):
        replogle_sample_weights[i] *= cfg.REPLOGLE_GWPS_DISCOUNT

# === Diagnostics ===
n_ess = sum(1 for n in replogle_names if n.startswith('ess_'))
n_gwps = sum(1 for n in replogle_names if n.startswith('gwps_'))
n_overlap_train = sum(1 for gi in replogle_gene_indices if gi in set(pert_gene_indices[pert_gene_indices >= 0]))

print(f'\n{"="*60}')
print(f'Replogle Augmentation Summary')
print(f'{"="*60}')
print(f'  Total perturbations:  {n_replogle:,}')
print(f'    Essential:          {n_ess:,}')
print(f'    GWPS:               {n_gwps:,}')
print(f'  Overlap with training: {n_overlap_train} / {len(pert_names)}')
print(f'  Novel genes:          {n_replogle - n_overlap_train:,}')
print(f'  Sample weight:        {cfg.REPLOGLE_SAMPLE_WEIGHT} (GWPS discount: ×{cfg.REPLOGLE_GWPS_DISCOUNT})')
print(f'  Mean |delta|:         {np.abs(replogle_deltas).mean():.6f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Replogle K562 Augmentation Diagnostics', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(['Essential', 'GWPS'], [n_ess, n_gwps], color=['#4CAF50', '#FF9800'])
ax.set_title(f'Replogle Source Breakdown (Total={n_replogle:,})')
ax.set_ylabel('# Perturbations')

ax = axes[1]
repl_abs = np.abs(replogle_deltas).mean(axis=1)
gt_abs = np.abs(deltas).mean(axis=1)
ax.hist(repl_abs, bins=60, alpha=0.6, color='#4CAF50', label=f'Replogle (n={n_replogle})', density=True)
ax.hist(gt_abs, bins=30, alpha=0.6, color='#E91E63', label=f'Competition GT (n={len(pert_names)})', density=True)
ax.set_title('Delta Magnitude: Replogle vs Competition')
ax.set_xlabel('Mean |Delta|')
ax.legend(fontsize=8)

ax = axes[2]
repl_ko = [replogle_deltas[i, replogle_gene_indices[i]] for i in range(n_replogle)
           if 0 <= replogle_gene_indices[i] < cfg.NUM_GENES]
gt_ko = [deltas[i, pert_gene_indices[i]] for i in range(len(pert_names))
         if 0 <= pert_gene_indices[i] < cfg.NUM_GENES]
ax.hist(repl_ko, bins=60, alpha=0.6, color='#4CAF50', label=f'Replogle KO ({len(repl_ko)})', density=True)
ax.hist(gt_ko, bins=30, alpha=0.6, color='#E91E63', label=f'Competition KO ({len(gt_ko)})', density=True)
ax.set_title('KO Self-Targeting Delta')
ax.set_xlabel('KO Gene Delta')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print('=' * 70)

In [ ]:
# ============================================================
# CELL 10: Model + Loss + Training Function (V20 — FROZEN)
#
# CASCADE NOTES:
# - LightMLP: V17 architecture (Embedding → 2-layer MLP → 5127)
# - cosine_light loss: WMAE - λ*cos (λ=0.08, cos_right=0.0405)
# - train_v20(): Full-batch training with GT + PB + LOO (+ optional Replogle)
# - get_fold_pb/loo/replogle(): fold-safe data getters
# ============================================================
print('=' * 70)
print('CELL 10: V20 Model + Loss + Training (FROZEN)')
print('=' * 70)

# ==================== LOSS FUNCTION ====================

def cosine_light_loss(pred, target, weights, sample_weights=None,
                      lam=None, cos_right=None):
    if lam is None: lam = cfg.COSINE_LAMBDA
    if cos_right is None: cos_right = cfg.COS_RIGHT
    per_sample = (weights * torch.abs(pred - target)).mean(dim=1)
    if sample_weights is not None:
        per_sample = per_sample * sample_weights
    w_mae = per_sample.mean()
    x = torch.maximum(pred.abs(), target.abs())
    t = (x / cos_right).clamp(0, 1)
    sw = t * t * (3 - 2 * t)
    cos = F.cosine_similarity(pred * sw, target * sw, dim=-1).mean()
    return w_mae - lam * cos

# ==================== MODEL (V17 LightMLP — FROZEN) ====================

class LightMLP(nn.Module):
    def __init__(self, n_genes, hidden=256, dropout=0.3, depth=2, embed_dim=64):
        super().__init__()
        self.n_genes = n_genes
        self.gene_embed = nn.Embedding(n_genes + 1, embed_dim)
        layers = []
        in_dim = embed_dim
        for _ in range(depth):
            layers.extend([nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden),
                          nn.ReLU(), nn.Dropout(dropout)])
            in_dim = hidden
        layers.append(nn.Linear(in_dim, n_genes))
        self.mlp = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.gene_embed.weight, std=0.02)

    def forward(self, gene_idx, mixup_embed=None):
        if mixup_embed is not None:
            return self.mlp(mixup_embed)
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.mlp(self.gene_embed(safe_idx))

    def get_embed(self, gene_idx):
        safe_idx = gene_idx.clone()
        safe_idx[safe_idx < 0] = self.n_genes
        safe_idx[safe_idx >= self.n_genes] = self.n_genes
        return self.gene_embed(safe_idx)

# ==================== TRAINING FUNCTION ====================

def train_v20(model, pb_indices, pb_deltas_arr, pb_weights_arr, pb_sample_w,
              gt_indices, gt_deltas_arr, gt_weights_arr, gt_upweight,
              loo_indices, loo_deltas_arr, loo_weights_arr, loo_sample_w,
              val_indices, val_deltas, val_weights, val_bl_wmae,
              repl_indices=None, repl_deltas_arr=None, repl_weights_arr=None, repl_sample_w=None,
              use_mixup=False, mixup_p=0.3, mixup_beta=0.2,
              weight_decay=None, epochs=None, lr=None, patience=None,
              device='cuda'):
    """V20 training: GT + PB + LOO + optional Replogle. Full-batch, no DataLoader."""
    epochs = epochs or cfg.MLP_EPOCHS
    lr = lr or cfg.MLP_LR
    patience = patience or cfg.EARLY_STOP
    wd = weight_decay if weight_decay is not None else cfg.WEIGHT_DECAY

    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr / 100)

    best_score, best_state, pat_count = -float('inf'), None, 0
    amp_on = USE_AMP and str(device).startswith('cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

    # Combine PB + GT + LOO + Replogle data
    parts_i = [pb_indices, gt_indices, loo_indices]
    parts_d = [pb_deltas_arr, gt_deltas_arr, loo_deltas_arr]
    parts_w = [pb_weights_arr, gt_weights_arr, loo_weights_arr]
    parts_sw = [
        pb_sample_w.astype(np.float32) if len(pb_sample_w) > 0 else np.array([], dtype=np.float32),
        np.full(len(gt_indices), gt_upweight, dtype=np.float32),
        loo_sample_w.astype(np.float32) if len(loo_sample_w) > 0 else np.array([], dtype=np.float32),
    ]

    if repl_indices is not None and len(repl_indices) > 0:
        parts_i.append(repl_indices)
        parts_d.append(repl_deltas_arr)
        parts_w.append(repl_weights_arr)
        parts_sw.append(repl_sample_w.astype(np.float32))

    all_i = np.concatenate(parts_i)
    all_d = np.concatenate(parts_d, axis=0).astype(np.float32)
    all_w = np.concatenate(parts_w, axis=0).astype(np.float32)
    all_sw = np.concatenate(parts_sw)

    va_i = torch.tensor(val_indices, dtype=torch.long, device=device)

    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(all_i))
        tr_i = torch.tensor(all_i[perm], dtype=torch.long, device=device)
        tr_d = torch.tensor(all_d[perm], dtype=torch.float32, device=device)
        tr_w = torch.tensor(all_w[perm], dtype=torch.float32, device=device)
        tr_sw = torch.tensor(all_sw[perm], dtype=torch.float32, device=device)

        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=amp_on):
            pred = model(tr_i)
            loss = cosine_light_loss(pred, tr_d, tr_w, tr_sw)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        scheduler.step()

        del tr_i, tr_d, tr_w, tr_sw

        # Validation every 10 epochs
        if (ep + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=amp_on):
                    vp = model(va_i).float().cpu().numpy()
            s, _, _ = metric.score(vp, val_deltas, val_weights, val_bl_wmae)
            if s > best_score:
                best_score = s
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                pat_count = 0
            else:
                pat_count += 10
            if pat_count >= patience:
                break

    if best_state: model.load_state_dict(best_state)

    del va_i; clean_memory()
    return model.cpu(), best_score


def get_fold_pb(val_pert_set):
    """Return PB samples excluding val fold perturbations."""
    mask = np.array([pn not in val_pert_set for pn in pb_pert_names])
    if mask.any():
        assert len(set(np.array(pb_pert_names)[mask]) & val_pert_set) == 0, 'LEAKAGE!'
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (pb_pert_indices[mask], pb_deltas[mask], pb_weights[mask],
            pb_cell_weights[mask])


def get_fold_loo(loo_gene_indices_arr, loo_deltas_arr, loo_sw_arr, val_pert_gene_set):
    """Return LOO samples excluding val fold gene indices (prevent leakage)."""
    mask = np.array([gi not in val_pert_gene_set for gi in loo_gene_indices_arr])
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    avg_w = weights_array.mean(axis=0)
    loo_w = np.tile(avg_w, (mask.sum(), 1))
    return (loo_gene_indices_arr[mask], loo_deltas_arr[mask], loo_w, loo_sw_arr[mask])


def get_fold_replogle(val_pert_gene_set):
    """Return Replogle samples excluding val fold gene indices."""
    mask = np.array([gi not in val_pert_gene_set for gi in replogle_gene_indices])
    if mask.sum() == 0:
        e = np.float32
        return (np.array([], dtype=np.int64), np.zeros((0, cfg.NUM_GENES), dtype=e),
                np.zeros((0, cfg.NUM_GENES), dtype=e), np.array([], dtype=e))
    return (replogle_gene_indices[mask], replogle_deltas[mask],
            replogle_weights[mask], replogle_sample_weights[mask])


# --- Model summary ---
test_model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                      dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                      embed_dim=cfg.EMBED_DIM)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'\nLightMLP params: {n_params:,}')
del test_model

print(f'Loss: cosine_light (lambda={cfg.COSINE_LAMBDA}, cos_right={cfg.COS_RIGHT})')
print(f'Training: train_v20() with GT + PB + LOO + optional Replogle, NO Mixup')
print('=' * 70)

In [ ]:
# ============================================================
# CELL 11: Multi-Run CV Engine with Checkpoint/Resume
#
# CASCADE NOTES:
# - Loops over RUN_CONFIGS (Runs 0-5)
# - Each run: 5-fold CV with fold-level checkpointing
# - Checkpoints saved to V23_DIR after each fold completes
# - On resume: skips already-completed folds
# - After all runs: adds Run 6 (best LOO + Replogle) dynamically
# - Produces rich per-fold and per-run diagnostics
# ============================================================
print('=' * 70)
print('CELL 11: Multi-Run CV Engine')
print('=' * 70)

all_run_results = {}
kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
fold_splits = list(kf.split(deltas))

t0_all = time.time()

for run_cfg in RUN_CONFIGS:
    run_id = run_cfg['id']
    run_name = run_cfg['name']
    loo_method = run_cfg['loo_method']
    use_replogle = run_cfg['replogle']

    # Check if run already completed
    existing_results = load_run_results(run_id)
    if existing_results is not None:
        print(f'\n{"="*60}')
        print(f'  Run {run_id} ({run_name}): ALREADY COMPLETED (CV={existing_results["cv_mean"]:.4f})')
        print(f'{"="*60}')
        all_run_results[run_id] = existing_results
        continue

    print(f'\n{"="*60}')
    print(f'  Run {run_id}: {run_name}')
    print(f'  LOO method: {loo_method.value} | Replogle: {use_replogle}')
    print(f'{"="*60}')

    # Get LOO data for this run
    loo_gi_all, loo_d_all = loo_cache[loo_method]
    loo_sw_all = np.full(len(loo_gi_all), cfg.LOO_SAMPLE_WEIGHT, dtype=np.float32)

    oof_preds = np.zeros_like(deltas, dtype=np.float32)
    fold_scores, fold_w_scores, fold_wcos_scores = [], [], []
    fold_times = []
    fold_data_sizes = []
    t0_run = time.time()

    for fold, (train_idx, val_idx) in enumerate(fold_splits):
        # Check checkpoint
        ckpt = load_checkpoint(run_id, fold)
        if ckpt is not None:
            preds_f, score_f, w_f, wcos_f = ckpt
            oof_preds[val_idx] = preds_f
            fold_scores.append(score_f)
            fold_w_scores.append(w_f)
            fold_wcos_scores.append(wcos_f)
            fold_times.append(0)
            fold_data_sizes.append({'gt': len(train_idx), 'pb': 0, 'loo': 0, 'repl': 0})
            continue

        fold_t0 = time.time()

        # --- Build fold data ---
        val_pert_set = {pert_names[i] for i in val_idx}
        val_gene_set = {pert_gene_indices[i] for i in val_idx if pert_gene_indices[i] >= 0}

        # GT
        gt_gi = pert_gene_indices[train_idx]
        gt_y = deltas[train_idx]
        gt_w = weights_array[train_idx]

        # PB (exclude val perts)
        pb_gi_f, pb_y_f, pb_w_f, pb_sw_f = get_fold_pb(val_pert_set)

        # LOO (exclude val gene indices)
        loo_gi_f, loo_y_f, loo_w_f, loo_sw_f = get_fold_loo(loo_gi_all, loo_d_all, loo_sw_all, val_gene_set)

        # Replogle (if enabled)
        repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = None, None, None, None
        n_repl = 0
        if use_replogle:
            repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = get_fold_replogle(val_gene_set)
            n_repl = len(repl_gi_f)

        # Validation
        val_gi = pert_gene_indices[val_idx]
        val_y = deltas[val_idx]
        val_w = weights_array[val_idx]
        val_bl = baseline_wmae[val_idx]

        n_gt, n_pb, n_loo = len(gt_gi), len(pb_gi_f), len(loo_gi_f)
        fold_data_sizes.append({'gt': n_gt, 'pb': n_pb, 'loo': n_loo, 'repl': n_repl})
        total_train = n_gt + n_pb + n_loo + n_repl
        print(f'\n  --- Run {run_id} Fold {fold+1}/{cfg.N_FOLDS} ---')
        print(f'  Train: {n_gt} GT + {n_pb} PB + {n_loo} LOO + {n_repl} Repl = {total_train}')
        print(f'  Val: {len(val_idx)} perts')

        # --- Ensemble training ---
        fold_ensemble_preds = np.zeros((len(val_idx), cfg.NUM_GENES), dtype=np.float32)

        for ens in range(cfg.N_MLP_ENSEMBLE):
            seed = cfg.SEED + run_id * 10000 + fold * 1000 + ens * 7
            torch.manual_seed(seed); np.random.seed(seed)

            model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                             dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                             embed_dim=cfg.EMBED_DIM)

            model, best_score = train_v20(
                model,
                pb_gi_f, pb_y_f, pb_w_f, pb_sw_f,
                gt_gi, gt_y, gt_w, cfg.GT_UPWEIGHT,
                loo_gi_f, loo_y_f, loo_w_f, loo_sw_f,
                val_gi, val_y, val_w, val_bl,
                repl_indices=repl_gi_f, repl_deltas_arr=repl_y_f,
                repl_weights_arr=repl_w_f, repl_sample_w=repl_sw_f,
                use_mixup=False,
                device=cfg.DEVICE)

            model.eval()
            with torch.no_grad():
                vp = model(torch.tensor(val_gi, dtype=torch.long)).numpy()
            fold_ensemble_preds += vp / cfg.N_MLP_ENSEMBLE
            del model; clean_memory()

        # --- KO Correction ---
        for i, vi in enumerate(val_idx):
            gi = pert_gene_indices[vi]
            if gi >= 0:
                fold_ensemble_preds[i] = apply_ko(fold_ensemble_preds[i], gi, knockout_stats, cfg.KNOCKOUT_MULTIPLIER)

        oof_preds[val_idx] = fold_ensemble_preds
        fold_score, fold_w, fold_wcos = metric.score(fold_ensemble_preds, val_y, val_w, val_bl)
        fold_scores.append(fold_score)
        fold_w_scores.append(fold_w)
        fold_wcos_scores.append(fold_wcos)

        elapsed = time.time() - fold_t0
        fold_times.append(elapsed)
        print(f'  Fold {fold+1}: {fold_score:.4f} (W={fold_w:.2f}, wcos={fold_wcos:.4f}) [{elapsed/60:.1f} min]')

        # Save checkpoint
        save_checkpoint(run_id, fold, fold_ensemble_preds, fold_score, fold_w, fold_wcos)

    # === Overall CV Score ===
    cv_score, cv_w, cv_wcos = metric.score(oof_preds, deltas, weights_array, baseline_wmae)
    cv_mean = float(np.mean(fold_scores))
    cv_std = float(np.std(fold_scores))
    run_time = time.time() - t0_run

    run_results = {
        'run_id': run_id, 'name': run_name,
        'loo_method': loo_method.value, 'replogle': use_replogle,
        'cv_mean': cv_mean, 'cv_std': cv_std,
        'cv_score': float(cv_score), 'cv_w': float(cv_w), 'cv_wcos': float(cv_wcos),
        'fold_scores': [float(s) for s in fold_scores],
        'fold_w': [float(w) for w in fold_w_scores],
        'fold_wcos': [float(w) for w in fold_wcos_scores],
        'fold_times': [float(t) for t in fold_times],
        'fold_data_sizes': fold_data_sizes,
        'total_time_min': run_time / 60,
    }

    save_run_results(run_id, run_name, run_results)
    all_run_results[run_id] = run_results

    print(f'\n  {"="*50}')
    print(f'  Run {run_id} ({run_name}) CV = {cv_mean:.4f} +/- {cv_std:.4f}')
    print(f'  OOF: {cv_score:.4f} (W={cv_w:.4f}, Wcos={cv_wcos:.4f})')
    print(f'  Folds: {[f"{s:.4f}" for s in fold_scores]}')
    print(f'  Time: {run_time/60:.1f} min')
    print(f'  {"="*50}')

# === After Runs 0-5: Add Run 6 (Best LOO + Replogle) ===
print(f'\n{"="*70}')
print('Determining best LOO method for Run 6...')

loo_run_ids = [0, 1, 2, 3, 4]
best_loo_run = max(loo_run_ids, key=lambda rid: all_run_results.get(rid, {}).get('cv_mean', -999))
best_loo_method = RUN_CONFIGS[best_loo_run]['loo_method']
best_loo_cv = all_run_results[best_loo_run]['cv_mean']

print(f'  Best LOO: Run {best_loo_run} ({best_loo_method.value}) with CV={best_loo_cv:.4f}')
print(f'  Run 6: {best_loo_method.value} + Replogle')

# Check if Run 6 already exists
run6_results = load_run_results(6)
if run6_results is not None:
    print(f'  Run 6 ALREADY COMPLETED (CV={run6_results["cv_mean"]:.4f})')
    all_run_results[6] = run6_results
else:
    # Execute Run 6
    run6_cfg = {"id": 6, "name": f"Best+Replogle ({best_loo_method.value})", "loo_method": best_loo_method, "replogle": True}
    loo_gi_all, loo_d_all = loo_cache[best_loo_method]
    loo_sw_all = np.full(len(loo_gi_all), cfg.LOO_SAMPLE_WEIGHT, dtype=np.float32)

    oof_preds = np.zeros_like(deltas, dtype=np.float32)
    fold_scores, fold_w_scores, fold_wcos_scores = [], [], []
    fold_times, fold_data_sizes = [], []
    t0_run = time.time()

    for fold, (train_idx, val_idx) in enumerate(fold_splits):
        ckpt = load_checkpoint(6, fold)
        if ckpt is not None:
            preds_f, score_f, w_f, wcos_f = ckpt
            oof_preds[val_idx] = preds_f
            fold_scores.append(score_f)
            fold_w_scores.append(w_f)
            fold_wcos_scores.append(wcos_f)
            fold_times.append(0)
            fold_data_sizes.append({'gt': len(train_idx), 'pb': 0, 'loo': 0, 'repl': 0})
            continue

        fold_t0 = time.time()
        val_pert_set = {pert_names[i] for i in val_idx}
        val_gene_set = {pert_gene_indices[i] for i in val_idx if pert_gene_indices[i] >= 0}

        gt_gi = pert_gene_indices[train_idx]
        gt_y = deltas[train_idx]
        gt_w = weights_array[train_idx]
        pb_gi_f, pb_y_f, pb_w_f, pb_sw_f = get_fold_pb(val_pert_set)
        loo_gi_f, loo_y_f, loo_w_f, loo_sw_f = get_fold_loo(loo_gi_all, loo_d_all, loo_sw_all, val_gene_set)
        repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = get_fold_replogle(val_gene_set)
        val_gi = pert_gene_indices[val_idx]
        val_y = deltas[val_idx]
        val_w = weights_array[val_idx]
        val_bl = baseline_wmae[val_idx]

        n_gt, n_pb, n_loo, n_repl = len(gt_gi), len(pb_gi_f), len(loo_gi_f), len(repl_gi_f)
        fold_data_sizes.append({'gt': n_gt, 'pb': n_pb, 'loo': n_loo, 'repl': n_repl})
        print(f'\n  --- Run 6 Fold {fold+1}/{cfg.N_FOLDS} ---')
        print(f'  Train: {n_gt} GT + {n_pb} PB + {n_loo} LOO + {n_repl} Repl = {n_gt+n_pb+n_loo+n_repl}')

        fold_ensemble_preds = np.zeros((len(val_idx), cfg.NUM_GENES), dtype=np.float32)
        for ens in range(cfg.N_MLP_ENSEMBLE):
            seed = cfg.SEED + 6 * 10000 + fold * 1000 + ens * 7
            torch.manual_seed(seed); np.random.seed(seed)
            model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                             dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH, embed_dim=cfg.EMBED_DIM)
            model, _ = train_v20(model,
                pb_gi_f, pb_y_f, pb_w_f, pb_sw_f,
                gt_gi, gt_y, gt_w, cfg.GT_UPWEIGHT,
                loo_gi_f, loo_y_f, loo_w_f, loo_sw_f,
                val_gi, val_y, val_w, val_bl,
                repl_indices=repl_gi_f, repl_deltas_arr=repl_y_f,
                repl_weights_arr=repl_w_f, repl_sample_w=repl_sw_f,
                use_mixup=False, device=cfg.DEVICE)
            model.eval()
            with torch.no_grad():
                vp = model(torch.tensor(val_gi, dtype=torch.long)).numpy()
            fold_ensemble_preds += vp / cfg.N_MLP_ENSEMBLE
            del model; clean_memory()

        for i, vi in enumerate(val_idx):
            gi = pert_gene_indices[vi]
            if gi >= 0:
                fold_ensemble_preds[i] = apply_ko(fold_ensemble_preds[i], gi, knockout_stats, cfg.KNOCKOUT_MULTIPLIER)

        oof_preds[val_idx] = fold_ensemble_preds
        fold_score, fold_w, fold_wcos = metric.score(fold_ensemble_preds, val_y, val_w, val_bl)
        fold_scores.append(fold_score)
        fold_w_scores.append(fold_w)
        fold_wcos_scores.append(fold_wcos)
        elapsed = time.time() - fold_t0
        fold_times.append(elapsed)
        print(f'  Fold {fold+1}: {fold_score:.4f} (W={fold_w:.2f}, wcos={fold_wcos:.4f}) [{elapsed/60:.1f} min]')
        save_checkpoint(6, fold, fold_ensemble_preds, fold_score, fold_w, fold_wcos)

    cv_score, cv_w, cv_wcos = metric.score(oof_preds, deltas, weights_array, baseline_wmae)
    cv_mean = float(np.mean(fold_scores))
    cv_std = float(np.std(fold_scores))
    run6_results = {
        'run_id': 6, 'name': run6_cfg['name'],
        'loo_method': best_loo_method.value, 'replogle': True,
        'cv_mean': cv_mean, 'cv_std': cv_std,
        'cv_score': float(cv_score), 'cv_w': float(cv_w), 'cv_wcos': float(cv_wcos),
        'fold_scores': [float(s) for s in fold_scores],
        'fold_w': [float(w) for w in fold_w_scores],
        'fold_wcos': [float(w) for w in fold_wcos_scores],
        'fold_times': [float(t) for t in fold_times],
        'fold_data_sizes': fold_data_sizes,
        'total_time_min': (time.time() - t0_run) / 60,
    }
    save_run_results(6, run6_cfg['name'], run6_results)
    all_run_results[6] = run6_results
    print(f'\n  Run 6 CV = {cv_mean:.4f} +/- {cv_std:.4f}')

total_time = (time.time() - t0_all) / 60
print(f'\n{"="*70}')
print(f'ALL RUNS COMPLETE — Total time: {total_time:.1f} min')
print(f'{"="*70}')
for rid in sorted(all_run_results.keys()):
    r = all_run_results[rid]
    print(f'  Run {rid:2d} ({r["name"]:25s}): CV={r["cv_mean"]:.4f} +/- {r["cv_std"]:.4f}  OOF={r["cv_score"]:.4f}')

In [ ]:
# ============================================================
# CELL 12: Comparison Dashboard — All Runs
#
# CASCADE NOTES:
# - Bar chart of CV scores across all 7 runs
# - Per-fold breakdown heatmap
# - Scatter: CV vs historical LB scores (inverse trend prediction)
# - Decision gate: select best run for final submission
# ============================================================
print('=' * 70)
print('CELL 12: V23 Comparison Dashboard')
print('=' * 70)

# Reload results in case of resumed session
for rid in range(7):
    if rid not in all_run_results:
        r = load_run_results(rid)
        if r is not None:
            all_run_results[rid] = r

completed_runs = sorted(all_run_results.keys())
n_runs = len(completed_runs)
print(f'Completed runs: {completed_runs}')

# === Figure 1: Main comparison ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('V23 Multi-Run Comparison Dashboard', fontsize=15, fontweight='bold')

# Chart 1: CV scores bar chart
ax = axes[0, 0]
run_labels = [f'R{rid}\n{all_run_results[rid]["name"][:12]}' for rid in completed_runs]
cv_means = [all_run_results[rid]['cv_mean'] for rid in completed_runs]
cv_stds = [all_run_results[rid]['cv_std'] for rid in completed_runs]
colors_run = ['#2196F3' if rid <= 4 else '#4CAF50' if rid == 5 else '#E91E63' for rid in completed_runs]
bars = ax.bar(run_labels, cv_means, yerr=cv_stds, capsize=4, color=colors_run, alpha=0.85, edgecolor='white')
ax.axhline(y=0.9672, color='green', linestyle='--', linewidth=1.5, label='V20 RUN=3 CV=0.967')
ax.axhline(y=0.9481, color='orange', linestyle='--', linewidth=1.5, label='V22 CV=0.948')
ax.set_title('CV Score per Run (mean ± std)')
ax.set_ylabel('CV Score')
ax.legend(fontsize=7, loc='lower right')
for bar, cv_m in zip(bars, cv_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{cv_m:.4f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

# Chart 2: Per-fold heatmap
ax = axes[0, 1]
fold_data = np.zeros((n_runs, cfg.N_FOLDS))
for i, rid in enumerate(completed_runs):
    scores = all_run_results[rid]['fold_scores']
    for f in range(min(len(scores), cfg.N_FOLDS)):
        fold_data[i, f] = scores[f]
im = ax.imshow(fold_data, aspect='auto', cmap='RdYlGn', vmin=fold_data.min() * 0.95, vmax=fold_data.max() * 1.02)
ax.set_xticks(range(cfg.N_FOLDS))
ax.set_xticklabels([f'Fold {f+1}' for f in range(cfg.N_FOLDS)], fontsize=8)
ax.set_yticks(range(n_runs))
ax.set_yticklabels([f'R{rid}' for rid in completed_runs], fontsize=8)
for i in range(n_runs):
    for j in range(cfg.N_FOLDS):
        ax.text(j, i, f'{fold_data[i,j]:.3f}', ha='center', va='center', fontsize=7,
                color='white' if fold_data[i,j] < fold_data.mean() else 'black')
ax.set_title('Per-Fold Scores (heatmap)')
plt.colorbar(im, ax=ax, shrink=0.6)

# Chart 3: CV vs predicted LB (inverse trend)
ax = axes[1, 0]
historical = {
    'V17 (0x LOO)': (0.866, 3.634),
    'V20 R4 (2x)': (0.981, 4.033),
    'V20 R3 (3x)': (0.967, 4.090),
    'V20 R8 (N40)': (0.969, 4.068),
    'V22 (4x+ch)': (0.948, 4.051),
    'V21 (sparse)': (0.967, 3.792),
}
for label, (cv, lb) in historical.items():
    ax.scatter(cv, lb, s=60, c='gray', marker='o', zorder=3, edgecolors='black', alpha=0.7)
    ax.annotate(label, (cv, lb), textcoords="offset points", xytext=(5, 5), fontsize=6)

# Plot V23 runs (CV only, LB unknown)
for rid in completed_runs:
    r = all_run_results[rid]
    ax.scatter(r['cv_mean'], 0, s=100, c=colors_run[completed_runs.index(rid)],
               marker='^', zorder=5, edgecolors='black')
    ax.annotate(f'R{rid}', (r['cv_mean'], 0), textcoords="offset points", xytext=(0, 8), fontsize=7)

ax.axhline(y=4.45, color='gold', linestyle='--', linewidth=1.5, label='#1: 4.45')
ax.set_title('CV vs LB (Historical + V23 Predictions)')
ax.set_xlabel('CV Score')
ax.set_ylabel('LB Score (V23 = unknown)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Chart 4: Summary & Decision
ax = axes[1, 1]
ax.axis('off')

best_run_id = max(completed_runs, key=lambda rid: all_run_results[rid]['cv_mean'])
best_r = all_run_results[best_run_id]
worst_run_id = min(completed_runs, key=lambda rid: all_run_results[rid]['cv_mean'])

summary_text = f'V23 Results Summary\n{"─"*50}\n'
for rid in completed_runs:
    r = all_run_results[rid]
    marker = ' ★' if rid == best_run_id else ''
    summary_text += f'  R{rid:2d} {r["name"]:25s} CV={r["cv_mean"]:.4f}±{r["cv_std"]:.4f}{marker}\n'

summary_text += f'\n{"─"*50}\n'
summary_text += f'BEST: Run {best_run_id} ({best_r["name"]}) CV={best_r["cv_mean"]:.4f}\n'
summary_text += f'  → This run will be used for final submission\n\n'

# Inverse trend prediction
# From V20: lower CV → higher LB (0.967 → 4.090, 0.948 → 4.051)
# This is imperfect but useful as a rough guide
if best_r['cv_mean'] < 0.967:
    summary_text += f'CV < V20 RUN=3 (0.967) → inverse trend predicts LB IMPROVEMENT\n'
elif best_r['cv_mean'] < 0.98:
    summary_text += f'CV ≈ V20 RUN=3 range → LB likely similar to 4.09\n'
else:
    summary_text += f'CV > 0.98 → less LOO regularization, LB might regress\n'

summary_text += f'\nTotal compute: {sum(all_run_results[r]["total_time_min"] for r in completed_runs):.0f} min'

ax.text(0.02, 0.98, summary_text, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()

# === Decision Gate ===
print(f'\n{"="*60}')
print(f'DECISION: Submit Run {best_run_id} ({best_r["name"]})')
print(f'  CV = {best_r["cv_mean"]:.4f}')
print(f'  LOO method: {best_r["loo_method"]}')
print(f'  Replogle: {best_r["replogle"]}')
print(f'{"="*60}')

BEST_RUN_ID = best_run_id
BEST_LOO_METHOD = LOOMethod(best_r['loo_method'])
BEST_USE_REPLOGLE = best_r['replogle']

In [ ]:
# ============================================================
# CELL 13: Final Training + Submission (Best Run)
#
# CASCADE NOTES:
# - Trains on ALL 80 GT + ALL PB + ALL LOO (+ Replogle if selected)
# - 90/10 early stopping split
# - N_MLP_ENSEMBLE=30 models averaged
# - KO correction ONLY (no hard thresholding)
# - Rows 61-120 zeroed (unscored test perts)
# - Saves submission to Drive + local for Kaggle upload
# - Also saves submissions for ALL 7 runs (not just best)
# ============================================================
print('=' * 70)
print(f'CELL 13: Final Training + Submission')
print(f'  Best run: {BEST_RUN_ID} ({all_run_results[BEST_RUN_ID]["name"]})')
print(f'  LOO: {BEST_LOO_METHOD.value} | Replogle: {BEST_USE_REPLOGLE}')
print('=' * 70)

def final_train_and_submit(run_id, loo_method, use_replogle, run_name):
    """Final training for one run config. Returns submission DataFrame."""
    sub_cache = os.path.join(V23_DIR, f'submission_run{run_id}.csv')
    if os.path.exists(sub_cache):
        print(f'  ✓ Run {run_id} submission already exists, loading...')
        return pd.read_csv(sub_cache)

    print(f'\n  --- Final Training: Run {run_id} ({run_name}) ---')
    loo_gi_all, loo_d_all = loo_cache[loo_method]
    loo_sw_all = np.full(len(loo_gi_all), cfg.LOO_SAMPLE_WEIGHT, dtype=np.float32)

    # 90/10 early stopping split
    n_es = max(8, len(pert_names) // 10)
    es_idx = np.random.RandomState(cfg.SEED).choice(len(pert_names), size=n_es, replace=False)
    tr_idx_final = np.array([i for i in range(len(pert_names)) if i not in es_idx])

    es_pert_set = {pert_names[i] for i in es_idx}
    es_gene_set = {pert_gene_indices[i] for i in es_idx if pert_gene_indices[i] >= 0}

    pb_i_f, pb_d_f, pb_w_f, pb_sw_f = get_fold_pb(es_pert_set)
    loo_gi_f, loo_y_f, loo_w_f, loo_sw_f = get_fold_loo(loo_gi_all, loo_d_all, loo_sw_all, es_gene_set)

    repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = None, None, None, None
    if use_replogle:
        repl_gi_f, repl_y_f, repl_w_f, repl_sw_f = get_fold_replogle(es_gene_set)

    tr_gi = pert_gene_indices[tr_idx_final]
    tr_y = deltas[tr_idx_final]
    tr_w = weights_array[tr_idx_final]
    es_gi = pert_gene_indices[es_idx]
    es_y = deltas[es_idx]
    es_w = weights_array[es_idx]
    es_bl = baseline_wmae[es_idx]

    n_repl = len(repl_gi_f) if repl_gi_f is not None else 0
    print(f'  Train: {len(tr_gi)} GT + {len(pb_i_f)} PB + {len(loo_gi_f)} LOO + {n_repl} Repl')
    print(f'  ES holdout: {n_es} perts')

    val_gi_list = list(val_gene_indices)

    t0 = time.time()
    final_preds = np.zeros((len(val_pert_ids), cfg.NUM_GENES), dtype=np.float32)

    for m_i in tqdm(range(cfg.N_MLP_ENSEMBLE), desc=f'R{run_id} training'):
        seed = cfg.SEED + run_id * 10000 + 9999 + m_i * 7
        torch.manual_seed(seed); np.random.seed(seed)

        model = LightMLP(cfg.NUM_GENES, hidden=cfg.MLP_HIDDEN,
                         dropout=cfg.MLP_DROPOUT, depth=cfg.MLP_DEPTH,
                         embed_dim=cfg.EMBED_DIM)

        model, _ = train_v20(
            model,
            pb_i_f, pb_d_f, pb_w_f, pb_sw_f,
            tr_gi, tr_y, tr_w, cfg.GT_UPWEIGHT,
            loo_gi_f, loo_y_f, loo_w_f, loo_sw_f,
            es_gi, es_y, es_w, es_bl,
            repl_indices=repl_gi_f, repl_deltas_arr=repl_y_f,
            repl_weights_arr=repl_w_f, repl_sample_w=repl_sw_f,
            use_mixup=False, device=cfg.DEVICE)

        model.eval()
        with torch.no_grad():
            p = model(torch.tensor(val_gi_list, dtype=torch.long)).numpy()
        final_preds += p
        del model; clean_memory()

    final_preds /= cfg.N_MLP_ENSEMBLE
    elapsed = (time.time() - t0) / 60

    # KO correction
    for pi in range(len(val_pert_ids)):
        gene_idx = val_gi_list[pi] if pi < len(val_gi_list) else -1
        final_preds[pi] = apply_ko(final_preds[pi], gene_idx, knockout_stats)

    # Rows 61-120 = zeros
    final_preds[60:] = 0.0

    # Build submission
    sub = pd.DataFrame(final_preds, columns=gene_names)
    sub.insert(0, 'pert_id', val_pert_ids)
    assert list(sub.columns) == SUBMISSION_COLUMNS
    assert len(sub) == len(val_pert_ids)
    assert not sub.isnull().any().any()

    # Save to Drive cache
    sub.to_csv(sub_cache, index=False)
    print(f'  ✓ Run {run_id} done in {elapsed:.1f} min, saved to cache')
    return sub

# === Train ALL runs (cached, so already-done runs load instantly) ===
all_submissions = {}
for rid in sorted(all_run_results.keys()):
    r = all_run_results[rid]
    sub_df = final_train_and_submit(
        rid, LOOMethod(r['loo_method']), r['replogle'], r['name'])
    all_submissions[rid] = sub_df

# === Save named submission files ===
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
saved_files = []

for rid in sorted(all_submissions.keys()):
    r = all_run_results[rid]
    cv_str = f'{r["cv_mean"]:.4f}'.replace('.', '_')
    method_str = r['loo_method']
    repl_str = '_repl' if r['replogle'] else ''
    fname = f'submission_v23_r{rid}_{method_str}{repl_str}_{ts}_{cv_str}.csv'

    # Save to Drive
    drive_path = os.path.join(OUTPUT_DIR, fname)
    all_submissions[rid].to_csv(drive_path, index=False)

    # Save locally for download
    local_path = f'/content/{fname}'
    all_submissions[rid].to_csv(local_path, index=False)

    saved_files.append((rid, fname, local_path))
    print(f'  Saved: {fname}')

# === Diagnostic Charts ===
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'V23 Final Submissions — Best: Run {BEST_RUN_ID}', fontsize=14, fontweight='bold')

# Chart 1: Prediction magnitude comparison across runs
ax = axes[0, 0]
for rid in sorted(all_submissions.keys()):
    preds = all_submissions[rid].iloc[:60, 1:].values.astype(np.float32)
    abs_means = np.abs(preds).mean(axis=1)
    ax.plot(range(60), abs_means, alpha=0.6, label=f'R{rid} ({all_run_results[rid]["name"][:12]})')
ax.set_title('Mean |Prediction| per Perturbation (scored rows)')
ax.set_xlabel('Perturbation Index')
ax.set_ylabel('Mean |Delta|')
ax.legend(fontsize=6, ncol=2)

# Chart 2: Pairwise correlation heatmap between run submissions
ax = axes[0, 1]
n_subs = len(all_submissions)
sub_ids = sorted(all_submissions.keys())
corr_mat = np.ones((n_subs, n_subs))
for i, ri in enumerate(sub_ids):
    for j, rj in enumerate(sub_ids):
        if i >= j: continue
        pi = all_submissions[ri].iloc[:60, 1:].values.ravel()
        pj = all_submissions[rj].iloc[:60, 1:].values.ravel()
        corr_mat[i, j] = corr_mat[j, i] = np.corrcoef(pi, pj)[0, 1]
im = ax.imshow(corr_mat, cmap='RdYlGn', vmin=0.98, vmax=1.0)
ax.set_xticks(range(n_subs)); ax.set_xticklabels([f'R{r}' for r in sub_ids], fontsize=8)
ax.set_yticks(range(n_subs)); ax.set_yticklabels([f'R{r}' for r in sub_ids], fontsize=8)
for i in range(n_subs):
    for j in range(n_subs):
        ax.text(j, i, f'{corr_mat[i,j]:.4f}', ha='center', va='center', fontsize=6)
ax.set_title('Submission Pairwise Correlation (scored rows)')
plt.colorbar(im, ax=ax, shrink=0.6)

# Chart 3: Sparsity comparison
ax = axes[1, 0]
for rid in sorted(all_submissions.keys()):
    preds = all_submissions[rid].iloc[:60, 1:].values.astype(np.float32)
    thresholds = np.linspace(0, 0.1, 100)
    pct_below = [100 * np.mean(np.abs(preds) < t) for t in thresholds]
    ax.plot(thresholds, pct_below, alpha=0.7, label=f'R{rid}')
ax.axvline(x=0.02, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Sparsity Profile per Run')
ax.set_xlabel('Threshold')
ax.set_ylabel('% Predictions Below')
ax.legend(fontsize=6, ncol=2)
ax.grid(True, alpha=0.3)

# Chart 4: Summary table
ax = axes[1, 1]
ax.axis('off')
table_text = f'V23 Submission Summary\n{"─"*55}\n'
table_text += f'{"Run":>4s} {"Name":>20s} {"CV":>8s} {"OOF":>8s} {"Repl":>5s}\n'
table_text += f'{"─"*55}\n'
for rid in sorted(all_run_results.keys()):
    r = all_run_results[rid]
    marker = ' ★' if rid == BEST_RUN_ID else ''
    table_text += f'  R{rid:<2d} {r["name"][:20]:>20s} {r["cv_mean"]:.4f}  {r["cv_score"]:.4f}  {"Y" if r["replogle"] else "N":>3s}{marker}\n'
table_text += f'{"─"*55}\n'
table_text += f'Best → Run {BEST_RUN_ID}: Submit this to Kaggle\n'
table_text += f'Timestamp: {ts}\n'
table_text += f'Files saved: {len(saved_files)}'

ax.text(0.02, 0.98, table_text, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.5))

plt.tight_layout()
plt.show()

# === Print download instructions ===
print(f'\n{"="*60}')
print('SUBMISSION FILES READY')
print(f'{"="*60}')
print(f'Best run for submission: Run {BEST_RUN_ID}')
best_file = [f for rid, f, _ in saved_files if rid == BEST_RUN_ID][0]
print(f'  File: {best_file}')
print(f'\nAll {len(saved_files)} submission files saved to Drive and /content/')
print('Upload the best to Kaggle, then fill in LB scores in Cell 14.')
print(f'{"="*60}')

In [ ]:
# ============================================================
# CELL 14: Post-Mortem & Next Steps
#
# CASCADE NOTES:
# Fill in LB scores after Kaggle submission for each run.
# This cell produces the final V23 analysis.
# ============================================================
print('=' * 70)
print('CELL 14: V23 Post-Mortem')
print('=' * 70)

# --- Fill in LB scores after submission ---
V23_LB_SCORES = {
    # Run ID: LB score (fill in after Kaggle submission)
    0: None,  # Baseline
    1: None,  # NN-Matched
    2: None,  # Residual
    3: None,  # Adaptive
    4: None,  # Weighted
    5: None,  # Replogle only
    6: None,  # Best LOO + Replogle
}

# Historical LB scores
lb_history = {
    'V6_best': 2.124, 'V14_cos': 3.021, 'V15.1': 3.419,
    'V16': 3.584, 'V17': 3.634, 'V18': 3.623,
    'V20_R4(2x)': 4.033, 'V20_R3(3x)': 4.090,
    'V20_R8(N40)': 4.068, 'V20_cv9642': 4.066,
    'V21_sparse': 3.792, 'V22_perch': 4.051,
}

# Add V23 scores
has_lb = False
for rid, lb in V23_LB_SCORES.items():
    if lb is not None:
        has_lb = True
        r = all_run_results.get(rid, {})
        lb_history[f'V23_R{rid}_{r.get("name","")[:8]}'] = lb

# === Charts ===
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('V23 Post-Mortem Analysis', fontsize=15, fontweight='bold')

# Chart 1: Full LB History
ax = axes[0, 0]
versions = list(lb_history.keys())
scores = list(lb_history.values())
colors_bar = ['#E91E63' if 'V23' in v else '#2196F3' if 'V20' in v
              else '#4CAF50' if 'V21' in v or 'V22' in v else '#9E9E9E' for v in versions]
ax.barh(versions, scores, color=colors_bar, alpha=0.8, edgecolor='white')
ax.axvline(x=4.45, color='gold', linestyle='--', linewidth=2, label='#1: 4.45')
ax.axvline(x=4.090, color='green', linestyle='--', linewidth=1.5, label='V20 best: 4.090')
ax.set_title('Full LB Score History')
ax.set_xlabel('LB Score')
ax.legend(fontsize=7, loc='lower right')
ax.set_xlim(left=1.5)

# Chart 2: CV vs LB for V23 runs
ax = axes[0, 1]
historical_cv_lb = [
    ('V17', 0.866, 3.634), ('V20 R4', 0.981, 4.033),
    ('V20 R3', 0.967, 4.090), ('V22', 0.948, 4.051),
    ('V21', 0.967, 3.792),
]
for label, cv, lb in historical_cv_lb:
    ax.scatter(cv, lb, s=60, c='gray', marker='o', edgecolors='black', alpha=0.7, zorder=3)
    ax.annotate(label, (cv, lb), textcoords="offset points", xytext=(5, 5), fontsize=7)

for rid, lb in V23_LB_SCORES.items():
    if lb is not None and rid in all_run_results:
        r = all_run_results[rid]
        ax.scatter(r['cv_mean'], lb, s=120, c='#E91E63', marker='*', edgecolors='black', zorder=5)
        ax.annotate(f'R{rid}', (r['cv_mean'], lb), textcoords="offset points", xytext=(5, 5), fontsize=8)

ax.axhline(y=4.45, color='gold', linestyle='--', linewidth=1.5, label='#1: 4.45')
ax.set_title('CV vs LB (V23 Runs)')
ax.set_xlabel('CV Score')
ax.set_ylabel('LB Score')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Chart 3: LOO method comparison (CV + LB)
ax = axes[1, 0]
loo_runs = [rid for rid in sorted(all_run_results.keys()) if rid <= 4]
if loo_runs:
    x_pos = np.arange(len(loo_runs))
    cv_vals = [all_run_results[rid]['cv_mean'] for rid in loo_runs]
    names_short = [all_run_results[rid]['name'][:10] for rid in loo_runs]

    bars_cv = ax.bar(x_pos - 0.15, cv_vals, 0.3, label='CV', color='#2196F3', alpha=0.8)
    ax.set_ylabel('CV Score', color='#2196F3')

    lb_vals = [V23_LB_SCORES.get(rid) for rid in loo_runs]
    if any(v is not None for v in lb_vals):
        ax2 = ax.twinx()
        lb_clean = [v if v is not None else 0 for v in lb_vals]
        bars_lb = ax2.bar(x_pos + 0.15, lb_clean, 0.3, label='LB', color='#E91E63', alpha=0.8)
        ax2.set_ylabel('LB Score', color='#E91E63')
        ax2.legend(loc='upper right', fontsize=8)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(names_short, fontsize=8)
    ax.set_title('LOO Method Comparison')
    ax.legend(loc='upper left', fontsize=8)

# Chart 4: Recommendations
ax = axes[1, 1]
ax.axis('off')

if has_lb:
    best_lb_rid = max((rid for rid, lb in V23_LB_SCORES.items() if lb is not None),
                      key=lambda r: V23_LB_SCORES[r])
    best_lb = V23_LB_SCORES[best_lb_rid]
    delta_vs_v20 = best_lb - 4.090

    analysis = f'V23 Results\n{"─"*45}\n'
    for rid, lb in sorted(V23_LB_SCORES.items()):
        if lb is not None:
            r = all_run_results.get(rid, {})
            analysis += f'  R{rid} ({r.get("name",""):>15s}): LB={lb:.5f}  CV={r.get("cv_mean",0):.4f}\n'
    analysis += f'\n{"─"*45}\n'
    analysis += f'Best V23: R{best_lb_rid} LB={best_lb:.5f}\n'
    analysis += f'vs V20 best (4.090): {delta_vs_v20:+.5f}\n'
    analysis += f'Gap to #1 (4.45):    {4.45 - best_lb:.3f}\n\n'

    if delta_vs_v20 > 0.01:
        analysis += 'IMPROVEMENT! New LOO quality method works.\n'
        analysis += 'V24 should: push resamples to 4x with best method\n'
    elif delta_vs_v20 > -0.01:
        analysis += 'Marginal change. LOO quality ~equivalent.\n'
        analysis += 'V24 should: try ensemble of V20+V23 runs\n'
    else:
        analysis += 'Regression. New methods hurt generalization.\n'
        analysis += 'V24 should: return to V20 baseline, try other axes\n'
else:
    analysis = (
        f'V23 Pre-Submission\n{"─"*45}\n'
        f'Fill in V23_LB_SCORES dict above\n'
        f'after submitting to Kaggle, then re-run.\n\n'
        f'CV Results:\n'
    )
    for rid in sorted(all_run_results.keys()):
        r = all_run_results[rid]
        analysis += f'  R{rid} ({r["name"]:>20s}): CV={r["cv_mean"]:.4f}\n'
    analysis += f'\nBest CV: Run {BEST_RUN_ID}\n'

ax.text(0.02, 0.98, analysis, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()

total_elapsed = sum(all_run_results[r].get('total_time_min', 0) for r in all_run_results)
print(f'\nTotal V23 compute: {total_elapsed:.0f} min')
print(f'Submission files: {len(saved_files)} saved')
if has_lb:
    print(f'Best LB: Run {best_lb_rid} = {best_lb}')
else:
    print('Fill in LB scores above and re-run for full analysis')
print('=' * 70)